# ClueLLM — Team Conniving Kangaroo

**Project template:** Template 1 — *Search-Based Drama Management*
**System name:** ClueLLM
**Phase:** II (final code)
**Members:** Devam Shrivastava, Chaeyoung Lee, David Lee

This notebook contains the complete system: Phase I generates the underlying
mystery (crime, characters, clues, 15 plot points, final narrative), and Phase II
turns that mystery into a playable, text-based interactive game governed by a
search-based drama manager.


### Technical Details

- **Model:** Google Gemini 2.5 Flash (`gemini-2.5-flash`)
- **API key:** read from `GOOGLE_API_KEY` (env var) or prompted via `getpass` — **never hardcoded**
- **Typical full-run time** (no interactive play): ~5–8 minutes
- **Typical cost:** ~$0.15–$0.25 per full run
- **Outputs of interest:** `world.final_narrative` (Phase I prose), `dm_log.json` (Phase II per-turn DM trace for the demo video)


In [1]:
!pip install -q -U google-genai

## LLM Setup

Define Gemini model API and LLM helper methods.

In [2]:
import os
import getpass

# API key handling (per rubric guidance):
# 1. Prefer GOOGLE_API_KEY from the environment / shell.
# 2. Fall back to a one-line interactive prompt (works in Jupyter).
# Never commit a real key into this notebook.
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your GOOGLE_API_KEY: ")
    except Exception:
        # Non-interactive context (e.g. CI / nbconvert without input) -- leave empty.
        os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY", "")

assert os.environ.get("GOOGLE_API_KEY"), (
    "GOOGLE_API_KEY is not set. Run `export GOOGLE_API_KEY=...` before launching Jupyter, "
    "or paste your key when prompted above."
)


Enter your GOOGLE_API_KEY:  ········


In [3]:
from google import genai

client = genai.Client()
MODEL = "gemini-2.5-flash"

# Sanity check
response = client.models.generate_content(model=MODEL, contents="Say: ready")
print("LLM status:", response.text.strip())

LLM status: ready


In [4]:
import time
import random as _rand


def ask_llm(prompt: str, *, max_attempts: int = 5, base_delay: float = 2.0) -> str:
    """Call the LLM with simple exponential backoff on transient server / rate
    errors. Gemini occasionally returns 503 UNAVAILABLE under load and 429 when
    quota is briefly exceeded; we retry those a few times rather than failing
    the whole notebook (rubric: code runs to completion upon execution)."""
    last_exc = None
    for attempt in range(max_attempts):
        try:
            response = client.models.generate_content(model=MODEL, contents=prompt)
            return response.text.strip()
        except Exception as exc:
            msg = str(exc)
            transient = any(s in msg for s in ("503", "429", "500", "UNAVAILABLE",
                                               "RESOURCE_EXHAUSTED", "DEADLINE_EXCEEDED"))
            last_exc = exc
            if not transient or attempt == max_attempts - 1:
                raise
            sleep_for = base_delay * (2 ** attempt) + _rand.uniform(0, 1)
            print(f"[ask_llm] transient error on attempt {attempt + 1}/{max_attempts}: "
                  f"{msg[:120]} -- retrying in {sleep_for:.1f}s")
            time.sleep(sleep_for)
    raise last_exc  # pragma: no cover


In [5]:
def extract_json(text: str):
    """
    Extract the first JSON object or array from model output.
    Handles markdown code fences like ```json ... ```.
    """
    # Remove markdown fences if present
    fenced = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()

    # Try full parse first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback: find first JSON object or array
    match = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    if match:
        return json.loads(match.group(1))

    raise ValueError(f"No JSON found in LLM response:\n{text}")

## Data Structures
Defines all dataclasses used to store story state. Sections 2 and 3 (story generation and compilation) will populate these.

| Class | Role | Function |
|---|---|---|
| `CrimeDetails` | The hidden truth | Keeps the ground-truth crime facts isolated so the LLM can never accidentally leak them into suspect dialogue or clue descriptions |
| `Character` | Any person in the story | Stores means/motive/opportunity explicitly so the meta-controller can check MMO coverage without re-querying the LLM |
| `Clue` | A piece of evidence | `is_red_herring` and `discovered` flags let the controller track what the detective *can* know vs. what *really* points to the criminal |
| `PlotPoint` | One story beat | `suspense_score` on each beat makes the rising tension arc a measurable, enforceable property rather than a vague stylistic goal |
| `StoryWorld` | Global state container | A single object passed by reference into every generation function — any LLM call can read current state via `summary_so_far()` without duplicating data |

In [6]:
import json
import re
from dataclasses import dataclass, field
from typing import List, Optional
from pprint import pprint


@dataclass
class Character:
    """A character in the story: detective, suspect, criminal, or witness."""
    name: str
    role: str                        # 'detective' | 'suspect' | 'criminal' | 'witness'
    description: str
    means: str                       # capability to commit the crime
    motive: str                      # reason to commit the crime
    opportunity: str                 # could they have been at the scene?
    alibi: str
    alibi_is_lie: bool = False
    is_criminal: bool = False
    eliminated: bool = False         # True once the detective rules them out
    notes: List[str] = field(default_factory=list)


@dataclass
class Clue:
    """A piece of evidence or information discoverable by the detective."""
    clue_id: int
    description: str
    location: str
    discovered: bool = False
    is_red_herring: bool = False
    points_to: str = ""              # character or event this implicates
    revealed_in_plot_point: int = -1


@dataclass
class PlotPoint:
    """One story beat that advances the plot."""
    index: int
    title: str
    description: str
    action_taken: str
    outcome: str
    obstacle: str
    suspense_score: float            # 0.0 (low tension) → 1.0 (maximum)
    characters_involved: List[str] = field(default_factory=list)
    clues_revealed: List[int] = field(default_factory=list)
    clues_used: List[int] = field(default_factory=list)


@dataclass
class CrimeDetails:
    """The hidden backstory: what really happened before the detective arrives."""
    crime_type: str
    victim: str
    location: str
    time_of_crime: str
    method: str
    criminal_name: str
    criminal_motive: str
    criminal_means: str
    key_evidence_hidden: List[str] = field(default_factory=list)
    backstory_summary: str = ""


@dataclass
class StoryWorld:
    """Master container for all story state."""
    crime_details: Optional[CrimeDetails] = None
    characters: List[Character] = field(default_factory=list)
    clues: List[Clue] = field(default_factory=list)
    plot_points: List[PlotPoint] = field(default_factory=list)
    countdown_deadline: str = ""
    countdown_consequence: str = ""
    final_narrative: str = ""

    def get_character(self, name: str) -> Optional[Character]:
        for c in self.characters:
            if c.name.lower() == name.lower():
                return c
        return None

    def get_clue(self, clue_id: int) -> Optional[Clue]:
        for cl in self.clues:
            if cl.clue_id == clue_id:
                return cl
        return None

    def discovered_clues(self) -> List[Clue]:
        return [cl for cl in self.clues if cl.discovered]

    def active_suspects(self) -> List[Character]:
        return [c for c in self.characters if c.role in ('suspect', 'criminal') and not c.eliminated]

    def summary_so_far(self) -> str:
        """Brief text digest of current story state, used as context in LLM prompts."""
        lines = []
        if self.crime_details:
            cd = self.crime_details
            lines.append(f"CRIME: {cd.crime_type} of {cd.victim} at {cd.location} ({cd.time_of_crime}).")
        lines.append(f"ACTIVE SUSPECTS: {[c.name for c in self.active_suspects()]}")
        lines.append(f"CLUES FOUND: {[cl.description[:40] for cl in self.discovered_clues()]}")
        if self.plot_points:
            last = self.plot_points[-1]
            lines.append(f"LAST PLOT POINT ({last.index}): {last.title} — {last.outcome}")
        return "\n".join(lines)


print("Data structures defined.")


Data structures defined.


In [7]:
# Initialize StoryWorld object
world = StoryWorld()
print("StoryWorld initialized:", world)


StoryWorld initialized: StoryWorld(crime_details=None, characters=[], clues=[], plot_points=[], countdown_deadline='', countdown_consequence='', final_narrative='')


## Phase 1: Crime Backstory

In [8]:
# Builds the prompt for the LLM to generate the crime details
def build_crime_details_prompt() -> str:
    return """
You are helping build the hidden ground-truth backstory for a gala murder crime mystery.

Generate a single crime scenario for a detective story.

Requirements:
- The crime must be a serious crime suitable for a mystery story.
- It must be solvable, but not obvious.
- The crime must support multiple plausible suspects later.
- The criminal's motive, means, and opportunity must all make sense together.
- The method of the crime must leave behind evidence that can later be discovered.
- The crime should not rely on supernatural elements or impossible coincidences.

Return ONLY valid JSON with exactly these keys:
{
  "crime_type": "...",
  "victim": "...",
  "location": "...",
  "time_of_crime": "...",
  "method": "...",
  "criminal_name": "...",
  "criminal_motive": "...",
  "criminal_means": "...",
  "key_evidence_hidden": ["...", "...", "..."],
  "backstory_summary": "..."
}

Field guidance:
- crime_type: e.g. "murder", "blackmail", "kidnapping", "art theft" (in this case, it will be "murder")
- victim: full name and a few identifying words
- location: specific and story-friendly
- time_of_crime: specific enough to anchor a timeline
- method: how the crime was carried out
- criminal_name: full name
- criminal_motive: concrete and personal
- criminal_means: how they were capable of doing it
- key_evidence_hidden: 3 to 5 important hidden evidence items
- backstory_summary: 1 paragraph summary of what truly happened before the detective starts investigating

Important:
- The criminal should not be the most obvious suspect.
- The motive should be meaningful, not generic.
- The evidence should connect naturally to the crime method.
- Return JSON only. No explanation.
""".strip()


# Prompts the LLM and extracts the answer to a data structure
def generate_crime_details() -> CrimeDetails:
    prompt = build_crime_details_prompt()
    raw = ask_llm(prompt)
    data = extract_json(raw)

    crime = CrimeDetails(
        crime_type=data["crime_type"],
        victim=data["victim"],
        location=data["location"],
        time_of_crime=data["time_of_crime"],
        method=data["method"],
        criminal_name=data["criminal_name"],
        criminal_motive=data["criminal_motive"],
        criminal_means=data["criminal_means"],
        key_evidence_hidden=data.get("key_evidence_hidden", []),
        backstory_summary=data.get("backstory_summary", "")
    )
    return crime

In [9]:
# Run crime detail generation
world.crime_details = generate_crime_details()

pprint(world.crime_details)
print()
print(world.crime_details.backstory_summary)

CrimeDetails(crime_type='murder',
             victim='Lord Alistair Finch, a notoriously ruthless real estate '
                    'magnate and philanthropist',
             location='The Grand Ballroom of the Étoile Hotel, during the '
                      "annual 'Starlight Charity Gala'",
             time_of_crime='Around 10:45 PM, shortly after the main auction '
                           'concluded',
             method='Lord Finch was poisoned. A fast-acting, concentrated '
                    'sedative (Xylazine) was meticulously frozen into a '
                    'specific, slightly larger ice cube. This ice cube was '
                    'then discreetly placed into his usual single-malt scotch, '
                    'served to him by a temporary waiter (unaware of the '
                    'poison) from a tray of drinks during a brief blackout '
                    "caused by a 'flicker' in the ballroom's lighting system. "
                    'The sedative acted quickl

## Phase 2: Characters & Clues

In [10]:
# Builds the prompt for generating character list
def build_characters_prompt(crime: CrimeDetails) -> str:
    crime_payload = {
        "crime_type": crime.crime_type,
        "victim": crime.victim,
        "location": crime.location,
        "time_of_crime": crime.time_of_crime,
        "method": crime.method,
        "criminal_name": crime.criminal_name,
        "criminal_motive": crime.criminal_motive,
        "criminal_means": crime.criminal_means,
        "key_evidence_hidden": crime.key_evidence_hidden,
        "backstory_summary": crime.backstory_summary,
    }

    return f"""
You are generating the cast of characters for a crime mystery.

The hidden ground-truth crime is:
{json.dumps(crime_payload, indent=2)}

Create:
- 1 detective
- 5 suspects who are NOT the criminal
- 1 criminal character whose name MUST exactly match the criminal_name above

Return ONLY valid JSON with exactly this structure:
{{
  "detective": {{
    "name": "...",
    "role": "detective",
    "description": "...",
    "means": "N/A",
    "motive": "solve the case",
    "opportunity": "arrives after the crime to investigate",
    "alibi": "N/A"
  }},
  "criminal": {{
    "name": "...",
    "role": "criminal",
    "description": "...",
    "means": "...",
    "motive": "...",
    "opportunity": "...",
    "alibi": "...",
    "alibi_is_lie": true
  }},
  "suspects": [
    {{
      "name": "...",
      "role": "suspect",
      "description": "...",
      "means": "...",
      "motive": "...",
      "opportunity": "...",
      "alibi": "...",
      "alibi_is_lie": false
    }}
  ]
}}

Requirements:
- The criminal name must exactly match the given criminal_name.
- There must be exactly 5 suspects in the suspects list.
- Every suspect must have plausible means, motive, and opportunity.
- At least 2 suspects should look strongly suspicious at first.
- At least 2 innocent suspects should have a false or misleading alibi.
- The criminal should not be the most obvious suspect.
- The detective should feel capable and specific, not generic.
- Suspects should have relationships to the victim, location, or circumstances of the crime.
- Keep all details realistic and internally consistent.

Return JSON only.
""".strip()


# Prompts the LLM and extracts the character data to our data structure
def generate_characters(world: StoryWorld) -> List[Character]:
    if not world.crime_details:
        raise ValueError("world.crime_details must exist before generating characters.")

    raw = ask_llm(build_characters_prompt(world.crime_details))
    data = extract_json(raw)

    characters = []

    detective_data = data["detective"]
    detective = Character(
        name=detective_data["name"],
        role="detective",
        description=detective_data["description"],
        means=detective_data.get("means", "N/A"),
        motive=detective_data.get("motive", "solve the case"),
        opportunity=detective_data.get("opportunity", "arrives after the crime to investigate"),
        alibi=detective_data.get("alibi", "N/A"),
        alibi_is_lie=False,
        is_criminal=False,
        eliminated=False,
        notes=[]
    )
    characters.append(detective)

    criminal_data = data["criminal"]
    criminal = Character(
        name=criminal_data["name"],
        role="criminal",
        description=criminal_data["description"],
        means=criminal_data["means"],
        motive=criminal_data["motive"],
        opportunity=criminal_data["opportunity"],
        alibi=criminal_data["alibi"],
        alibi_is_lie=criminal_data.get("alibi_is_lie", True),
        is_criminal=True,
        eliminated=False,
        notes=[]
    )
    characters.append(criminal)

    for suspect_data in data["suspects"]:
        suspect = Character(
            name=suspect_data["name"],
            role="suspect",
            description=suspect_data["description"],
            means=suspect_data["means"],
            motive=suspect_data["motive"],
            opportunity=suspect_data["opportunity"],
            alibi=suspect_data["alibi"],
            alibi_is_lie=suspect_data.get("alibi_is_lie", False),
            is_criminal=False,
            eliminated=False,
            notes=[]
        )
        characters.append(suspect)

    return characters


# Builds the prompt for the LLM to validate our character list
def build_character_validation_prompt(world: StoryWorld, characters: List[Character]) -> str:
    crime = world.crime_details
    if not crime:
        raise ValueError("No crime details in world.")

    payload = {
        "crime_details": {
            "crime_type": crime.crime_type,
            "victim": crime.victim,
            "location": crime.location,
            "time_of_crime": crime.time_of_crime,
            "method": crime.method,
            "criminal_name": crime.criminal_name,
            "criminal_motive": crime.criminal_motive,
            "criminal_means": crime.criminal_means,
            "backstory_summary": crime.backstory_summary,
        },
        "characters": [
            {
                "name": c.name,
                "role": c.role,
                "description": c.description,
                "means": c.means,
                "motive": c.motive,
                "opportunity": c.opportunity,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
                "is_criminal": c.is_criminal,
            }
            for c in characters
        ]
    }

    return f"""
You are validating the character set for a crime mystery.

Data:
{json.dumps(payload, indent=2)}

Check:
1. Is there exactly 1 detective?
2. Is there exactly 1 criminal and does their name match the hidden criminal_name?
3. Are there exactly 5 non-criminal suspects?
4. Does every suspect have plausible means, motive, or opportunity?
5. Are at least 2 suspects genuinely suspicious at first?
6. Are the characters distinct from one another?
7. Are there any contradictions with the crime setup?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."]
}}
""".strip()


# Prompts the LLM to validate our character list
def validate_characters(world: StoryWorld, characters: List[Character]) -> dict:
    raw = ask_llm(build_character_validation_prompt(world, characters))
    return extract_json(raw)

In [11]:
# Builds the prompt for generating clues
def build_clues_prompt(world: StoryWorld) -> str:
    if not world.crime_details:
        raise ValueError("world.crime_details must exist before generating clues.")
    if not world.characters:
        raise ValueError("world.characters must exist before generating clues.")

    crime = world.crime_details
    character_payload = [
        {
            "name": c.name,
            "role": c.role,
            "description": c.description,
            "means": c.means,
            "motive": c.motive,
            "opportunity": c.opportunity,
            "alibi": c.alibi,
            "alibi_is_lie": c.alibi_is_lie,
            "is_criminal": c.is_criminal,
        }
        for c in world.characters
    ]

    crime_payload = {
        "crime_type": crime.crime_type,
        "victim": crime.victim,
        "location": crime.location,
        "time_of_crime": crime.time_of_crime,
        "method": crime.method,
        "criminal_name": crime.criminal_name,
        "criminal_motive": crime.criminal_motive,
        "criminal_means": crime.criminal_means,
        "key_evidence_hidden": crime.key_evidence_hidden,
        "backstory_summary": crime.backstory_summary,
    }

    return f"""
You are generating discoverable clues for a crime mystery.

Hidden crime details:
{json.dumps(crime_payload, indent=2)}

Character list:
{json.dumps(character_payload, indent=2)}

Generate 8 clues.

Return ONLY valid JSON with exactly this structure:
{{
  "clues": [
    {{
      "clue_id": 1,
      "description": "...",
      "location": "...",
      "is_red_herring": false,
      "points_to": "...",
      "why_it_matters": "..."
    }}
  ]
}}

Requirements:
- Generate exactly 8 clues.
- Clues must follow naturally from the true crime and the characters.
- At least 2 clues must initially point toward innocent suspects.
- At least 2 clues must meaningfully connect to the real criminal.
- At least 1 clue should relate to a false alibi.
- At least 1 clue should be ambiguous until later interpretation.
- The clues should not instantly solve the crime when seen alone.
- "points_to" should name a character or a specific event/fact.
- clue_id values must be 1 through 8.

Return JSON only.
""".strip()


# Prompts the LLM and extracts clue data to our data structure
def generate_clues(world: StoryWorld) -> List[Clue]:
    raw = ask_llm(build_clues_prompt(world))
    data = extract_json(raw)

    clues = []
    for clue_data in data["clues"]:
        clue = Clue(
            clue_id=clue_data["clue_id"],
            description=clue_data["description"],
            location=clue_data["location"],
            discovered=False,
            is_red_herring=clue_data.get("is_red_herring", False),
            points_to=clue_data.get("points_to", ""),
            revealed_in_plot_point=-1
        )
        clues.append(clue)

    return clues


# Builds the prompt for the LLM to validate our clues
def build_clue_validation_prompt(world: StoryWorld, clues: List[Clue]) -> str:
    crime = world.crime_details
    if not crime:
        raise ValueError("No crime details in world.")

    payload = {
        "crime_details": {
            "crime_type": crime.crime_type,
            "victim": crime.victim,
            "location": crime.location,
            "time_of_crime": crime.time_of_crime,
            "method": crime.method,
            "criminal_name": crime.criminal_name,
            "criminal_motive": crime.criminal_motive,
            "criminal_means": crime.criminal_means,
            "backstory_summary": crime.backstory_summary,
        },
        "characters": [
            {
                "name": c.name,
                "role": c.role,
                "is_criminal": c.is_criminal,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
            }
            for c in world.characters
        ],
        "clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
            }
            for cl in clues
        ]
    }

    return f"""
You are validating the clue set for a crime mystery.

Data:
{json.dumps(payload, indent=2)}

Check:
1. Are there exactly 8 clues?
2. Do the clues arise naturally from the crime and character setup?
3. Do at least 2 clues point toward innocent suspects at first?
4. Do at least 2 clues connect meaningfully to the real criminal?
5. Is the mystery neither trivial nor impossible?
6. Are any clues redundant, contradictory, or disconnected?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."]
}}
""".strip()


# Prompts the LLM to validate our clue list
def validate_clues(world: StoryWorld, clues: List[Clue]) -> dict:
    raw = ask_llm(build_clue_validation_prompt(world, clues))
    return extract_json(raw)

In [12]:
# Builds the prompt to generate a countdown deadline and consequence for our solving story countdown mechanism
def build_countdown_prompt(world: StoryWorld) -> str:
    if not world.crime_details:
        raise ValueError("world.crime_details must exist first")

    crime = world.crime_details

    payload = {
        "crime_type": crime.crime_type,
        "victim": crime.victim,
        "location": crime.location,
        "time_of_crime": crime.time_of_crime,
        "method": crime.method,
        "criminal_name": crime.criminal_name,
        "criminal_motive": crime.criminal_motive,
        "backstory_summary": crime.backstory_summary,
    }

    return f"""
You are designing a countdown mechanism for a suspenseful detective story.

The countdown must:
- Be concrete and time-bound
- Be directly connected to the crime or investigation
- Create urgency that increases over time
- Make failure meaningful and irreversible

Crime context:
{json.dumps(payload, indent=2)}

Generate:
1. A specific countdown deadline (what event is approaching?)
2. A specific consequence if the detective fails before the deadline

Constraints:
- The deadline should feel natural to the scenario (not arbitrary)
- The consequence must be severe and story-relevant
- The consequence should make the truth harder or impossible to recover
- Avoid vague phrasing like "things get worse"

Examples of good countdowns:
- "By sunrise, the crime scene will be destroyed by the incoming storm."
- "In 24 hours, the prosecutor will formally charge the wrong suspect."
- "The suspect is scheduled to leave the country tonight."
- "A building demolition will erase crucial evidence."

Do not copy these exactly. Generate a new one specific to the crime.

Return ONLY valid JSON:
{{
  "countdown_deadline": "...",
  "countdown_consequence": "..."
}}
""".strip()


# Prompts the LLM and extracts data to StoryWorld data structure
def generate_countdown(world: StoryWorld) -> StoryWorld:
    raw = ask_llm(build_countdown_prompt(world))
    data = extract_json(raw)

    world.countdown_deadline = data["countdown_deadline"]
    world.countdown_consequence = data["countdown_consequence"]

    return world

In [13]:
# Function to populate StoryWorld data structure with characters, clues, and countdown mechanism
# Also runs validation functions of our generated characters and clues and outputs if the LLM detects an issue
def populate_story_world(world: StoryWorld) -> StoryWorld:
    if not world.crime_details:
        raise ValueError("Generate crime details first.")

    characters = generate_characters(world)
    char_validation = validate_characters(world, characters)
    if not char_validation["is_valid"]:
        print("Invalid characters -- try again")
        pprint(char_validation)

    world.characters = characters

    clues = generate_clues(world)
    clue_validation = validate_clues(world, clues)
    if not clue_validation["is_valid"]:
        print("Invalid clues -- try again")
        pprint(clue_validation)

    world.clues = clues

    world = generate_countdown(world)
    return world

In [14]:
def print_characters(world: StoryWorld):
    print("\nCHARACTERS")
    print("=" * 60)
    for c in world.characters:
        print(f"Name: {c.name}")
        print(f"Role: {c.role}")
        print(f"Description: {c.description}")
        print(f"Means: {c.means}")
        print(f"Motive: {c.motive}")
        print(f"Opportunity: {c.opportunity}")
        print(f"Alibi: {c.alibi}")
        print(f"Alibi is lie: {c.alibi_is_lie}")
        print(f"Is criminal: {c.is_criminal}")
        print("-" * 60)


def print_clues(world: StoryWorld):
    print("\nCLUES")
    print("=" * 60)
    for cl in sorted(world.clues, key=lambda x: x.clue_id):
        print(f"Clue #{cl.clue_id}")
        print(f"Description: {cl.description}")
        print(f"Location: {cl.location}")
        print(f"Points to: {cl.points_to}")
        print(f"Red herring: {cl.is_red_herring}")
        print("-" * 60)


def print_countdown(world: StoryWorld):
    print("\nCOUNTDOWN SETUP")
    print("=" * 60)
    print(f"Deadline: {world.countdown_deadline}")
    print(f"Consequence: {world.countdown_consequence}")

In [15]:
# Runner cell to populate story world with characters, clues, and countdown mechanism
world = populate_story_world(world)

print_characters(world)
print_clues(world)
print_countdown(world)


CHARACTERS
Name: Detective Inspector Aris Thorne
Role: detective
Description: A seasoned detective inspector with a reputation for meticulous investigation and a keen understanding of human psychology. He's known for his quiet intensity and ability to uncover hidden motives, often preferring to observe and listen rather than dominate a scene. He arrived at the Étoile Hotel shortly after Finch's death was reported, immediately taking charge of the complex crime scene.
Means: N/A
Motive: solve the case
Opportunity: arrives after the crime to investigate
Alibi: N/A
Alibi is lie: False
Is criminal: False
------------------------------------------------------------
Name: Evelyn Reed
Role: criminal
Description: The Étoile Hotel's highly organized and seemingly unflappable lead caterer. Known for her impeccable events and calm demeanor, beneath her professional facade lies a deeply embittered individual consumed by a long-simmering desire for retribution against Lord Finch. Her outward compo

## Phase 3: Iterative Loop / Meta Controller

In [16]:
# Helper function for countdown mechanism
def build_countdown_text(step_index: int, total_steps: int = 15) -> str:
    remaining = total_steps - step_index + 1

    if remaining > 10:
        urgency = "low but rising"
    elif remaining > 5:
        urgency = "moderate and increasingly serious"
    elif remaining > 2:
        urgency = "high"
    else:
        urgency = "extreme"

    return (
        f"COUNTDOWN: The detective has only {remaining} major investigative moves remaining "
        f"before the case reaches a point of no return. Urgency level: {urgency}."
    )

In [17]:
# Summarizes plot so far to pass as context to next LLM prompt
def build_plot_context(world: StoryWorld) -> str:
    if not world.crime_details:
        raise ValueError("world.crime_details is missing")

    crime = world.crime_details

    detective = next((c for c in world.characters if c.role == "detective"), None)
    if detective is None:
        raise ValueError("No detective found in world.characters")

    suspects = [c for c in world.characters if c.role in ("suspect", "criminal")]
    discovered_clues = world.discovered_clues()

    payload = {
        "crime_details": {
            "crime_type": crime.crime_type,
            "victim": crime.victim,
            "location": crime.location,
            "time_of_crime": crime.time_of_crime,
            "method": crime.method,
            "backstory_summary": crime.backstory_summary,
        },
        "detective": {
            "name": detective.name,
            "description": detective.description,
            "motive": detective.motive,
        },
        "active_suspects": [
            {
                "name": c.name,
                "role": c.role,
                "description": c.description,
                "means": c.means,
                "motive": c.motive,
                "opportunity": c.opportunity,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
                "eliminated": c.eliminated,
            }
            for c in suspects if not c.eliminated
        ],
        "discovered_clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
            }
            for cl in discovered_clues
        ],
        "undiscovered_clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
            }
            for cl in world.clues if not cl.discovered
        ],
        "prior_plot_points": [
            {
                "index": p.index,
                "title": p.title,
                "description": p.description,
                "action_taken": p.action_taken,
                "outcome": p.outcome,
                "obstacle": p.obstacle,
                "suspense_score": p.suspense_score,
                "characters_involved": p.characters_involved,
                "clues_revealed": p.clues_revealed,
                "clues_used": p.clues_used,
            }
            for p in world.plot_points
        ]
    }

    return json.dumps(payload, indent=2)

In [18]:
# Prompt for generating next plot point
def build_next_plot_point_prompt(world: StoryWorld, step_index: int, total_steps: int = 15) -> str:
    detective = next((c for c in world.characters if c.role == "detective"), None)
    if detective is None:
        raise ValueError("No detective found")

    countdown_text = build_countdown_text(step_index, total_steps)

    detective_goal = (
        f"{detective.name} must identify the true criminal behind the "
        f"{world.crime_details.crime_type} of {world.crime_details.victim} before the final countdown expires."
    )

    deadline = getattr(world, "countdown_deadline", "")
    consequence = getattr(world, "countdown_consequence", "")

    countdown_block = f"""
    DEADLINE:
    {deadline}

    CONSEQUENCE:
    {consequence}
    """

    return f"""
You are the suspense-planning controller for a crime mystery.

Your job is to generate exactly ONE next plot point in a 15-step detective story.

This story must use suspense generation:
1. The audience cares about the detective and the victim.
2. The detective has a concrete goal.
3. Failure has dire consequences.
4. Each new plot point makes success harder, more urgent, or less likely.
5. Suspense comes from narrowing options, increasing uncertainty, and reducing chances to avert failure.
6. Do NOT resolve the full case early.

Detective goal:
{detective_goal}

{countdown_block}

{countdown_text}

Current story state:
{build_plot_context(world)}

Rules for this next plot point:
- This is plot point #{step_index} of {total_steps}.
- The detective must take a concrete action.
- That action must produce a meaningful outcome, but not fully solve the case unless this is the final plot point.
- Introduce or intensify an obstacle.
- Suspense must generally rise over time.
- Reveal 0, 1, or 2 clues at this step.
- Use already discovered clues when relevant.
- A clue can be revealed only once.
- You may eliminate at most one suspect in a single step.
- The detective should not identify the real criminal before plot point 13.
- Plot points 13-15 should feel like endgame escalation.
- A false lead, contradiction, missing evidence, time pressure, or witness unreliability can be used as obstacles.
- Keep the action grounded in crime solving, not action-movie spectacle.

Return ONLY valid JSON with exactly this structure:
{{
  "index": {step_index},
  "title": "...",
  "description": "...",
  "action_taken": "...",
  "outcome": "...",
  "obstacle": "...",
  "suspense_score": 0.0,
  "characters_involved": ["..."],
  "clues_revealed": [1, 2],
  "clues_used": [1, 2],
  "suspect_eliminated": "Name or empty string",
  "countdown_note": "How the countdown pressure changed in this step"
}}

Constraints on suspense_score:
- It should usually increase over time.
- Early steps: around 0.25 to 0.45
- Middle steps: around 0.45 to 0.75
- Late steps: around 0.75 to 0.98

Return JSON only.
""".strip()

In [19]:
# Integrate/apply new plot point to story
def apply_plot_point_to_world(world: StoryWorld, data: dict, step_index: int = -1) -> PlotPoint:
    # Be defensive: the LLM occasionally omits text fields. Fall back to sensible
    # defaults so the pipeline does not crash mid-generation.
    title = (data.get("title") or "").strip() or f"Plot Point {data.get('index', step_index)}"
    description = (data.get("description") or "").strip() or (data.get("outcome") or "").strip() or title
    action_taken = (data.get("action_taken") or "").strip() or "The detective continues investigating."
    outcome = (data.get("outcome") or "").strip() or description
    obstacle = (data.get("obstacle") or "").strip() or "An unresolved question lingers."

    plot_point = PlotPoint(
        index=int(data.get("index", step_index)),
        title=title,
        description=description,
        action_taken=action_taken,
        outcome=outcome,
        obstacle=obstacle,
        suspense_score=float(data.get("suspense_score", 0.5)),
        characters_involved=data.get("characters_involved", []),
        clues_revealed=data.get("clues_revealed", []),
        clues_used=data.get("clues_used", [])
    )

    # Mark newly revealed clues as discovered
    for clue_id in plot_point.clues_revealed:
        clue = world.get_clue(clue_id)
        if clue:
            clue.discovered = True
            clue.revealed_in_plot_point = plot_point.index

    # Eliminate suspect if applicable
    suspect_name = data.get("suspect_eliminated", "").strip()
    if suspect_name:
        character = world.get_character(suspect_name)
        if character and character.role in ("suspect", "criminal") and not character.is_criminal:
            character.eliminated = True
            character.notes.append(f"Eliminated in plot point {plot_point.index}")

    world.plot_points.append(plot_point)
    return plot_point


In [20]:
# Prompt for validating plot construction
def build_plot_point_validation_prompt(world: StoryWorld, candidate: dict, total_steps: int = 15) -> str:
    return f"""
You are validating one plot point in a suspenseful detective story.

Current story state:
{build_plot_context(world)}

Candidate plot point:
{json.dumps(candidate, indent=2)}

Check:
1. Is the detective taking a concrete investigative action?
2. Does the plot point increase or maintain suspense appropriately?
3. Does it avoid solving the whole case too early?
4. Are clue reveals valid and non-contradictory?
5. Is any suspect elimination justified?
6. Is the obstacle meaningful?
7. Is this consistent with a countdown-based suspense structure?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."]
}}
""".strip()


def validate_plot_point(world: StoryWorld, candidate: dict) -> dict:
    raw = ask_llm(build_plot_point_validation_prompt(world, candidate))
    return extract_json(raw)

In [21]:
# Prompts LLM with build_next_plot_point_prompt, validates, and integrates to story world object
def generate_next_plot_point(world: StoryWorld, step_index: int, total_steps: int = 15, max_attempts: int = 3) -> PlotPoint:
    last_candidate = None

    # Validation / retry mechanism
    for attempt in range(max_attempts):
        prompt = build_next_plot_point_prompt(world, step_index, total_steps)
        raw = ask_llm(prompt)
        candidate = extract_json(raw)
        last_candidate = candidate

        validation = validate_plot_point(world, candidate)

        if validation.get("is_valid", False):
            return apply_plot_point_to_world(world, candidate)

    # fallback: use the last candidate even if imperfect
    if last_candidate is None:
        raise ValueError(f"Failed to generate plot point {step_index}")

    return apply_plot_point_to_world(world, last_candidate)

### Meta Controller

In [22]:
# Meta controller that progresses story by introducing plot points and injecting suspense
def generate_plot_points(world: StoryWorld, total_steps: int = 15) -> StoryWorld:
    if not world.crime_details:
        raise ValueError("Generate crime details first.")
    if not world.characters:
        raise ValueError("Generate characters first.")
    if not world.clues:
        raise ValueError("Generate clues first.")

    world.plot_points = []

    for step_index in range(1, total_steps + 1):
        print(f"Generating plot point {step_index}/{total_steps}...")
        plot_point = generate_next_plot_point(world, step_index, total_steps)
        print(f"  -> {plot_point.title} (suspense={plot_point.suspense_score:.2f})")

    return world

In [23]:
# Fully validates plot post-generation
def build_full_plot_validation_prompt(world: StoryWorld) -> str:
    payload = {
        "crime_type": world.crime_details.crime_type if world.crime_details else "",
        "victim": world.crime_details.victim if world.crime_details else "",
        "plot_points": [
            {
                "index": p.index,
                "title": p.title,
                "description": p.description,
                "action_taken": p.action_taken,
                "outcome": p.outcome,
                "obstacle": p.obstacle,
                "suspense_score": p.suspense_score,
                "characters_involved": p.characters_involved,
                "clues_revealed": p.clues_revealed,
                "clues_used": p.clues_used,
            }
            for p in world.plot_points
        ],
        "remaining_active_suspects": [c.name for c in world.active_suspects()],
        "discovered_clues": [cl.clue_id for cl in world.discovered_clues()]
    }

    return f"""
You are validating the full suspense arc of a 15-point detective mystery.

Data:
{json.dumps(payload, indent=2)}

Check:
1. Do the 15 plot points form a coherent progression?
2. Does suspense generally rise?
3. Does the countdown mechanism feel meaningful?
4. Is the case not solved too early?
5. Are the final 3 plot points endgame escalation?
6. Are clue reveals paced well?
7. Are there any repetitive or filler beats?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."],
  "arc_score": 0.0
}}
""".strip()


def validate_full_plot_arc(world: StoryWorld) -> dict:
    raw = ask_llm(build_full_plot_validation_prompt(world))
    return extract_json(raw)

In [24]:
def print_plot_points(world: StoryWorld):
    print("\nPLOT POINTS")
    print("=" * 80)

    for p in sorted(world.plot_points, key=lambda x: x.index):
        print(f"[{p.index}] {p.title}")
        print(f"Description: {p.description}")
        print(f"Action Taken: {p.action_taken}")
        print(f"Outcome: {p.outcome}")
        print(f"Obstacle: {p.obstacle}")
        print(f"Suspense Score: {p.suspense_score:.2f}")
        print(f"Characters Involved: {p.characters_involved}")
        print(f"Clues Revealed: {p.clues_revealed}")
        print(f"Clues Used: {p.clues_used}")
        print("-" * 80)

In [25]:
# Runner cell to generate plot points and validate story

world = generate_plot_points(world, total_steps=15)
print_plot_points(world)

arc_validation = validate_full_plot_arc(world)
print("\nFULL ARC VALIDATION")
pprint(arc_validation)

Generating plot point 1/15...
  -> A Flicker in the Dark, a Peculiar Ice Cube (suspense=0.35)
Generating plot point 2/15...
  -> A Chilling Find in Cold Storage (suspense=0.42)
Generating plot point 3/15...
  -> A Sedative's Whisper in the Cold (suspense=0.58)
Generating plot point 4/15...
  -> The Judge's Tightened Grip (suspense=0.68)
Generating plot point 5/15...
  -> Encrypted Secrets, Judicial Delays (suspense=0.75)
Generating plot point 6/15...
  -> A Rival's Heated Threat (suspense=0.80)
Generating plot point 7/15...
  -> A Rival's Alibi, a Shortened Fuse (suspense=0.88)
Generating plot point 8/15...
  -> A Silent Switch, A Lingering Shadow (suspense=0.92)
Generating plot point 9/15...
  -> The Assistant's Faltered Alibi (suspense=0.94)
Generating plot point 10/15...
  -> The Orchestrator's Device, The Judge's Doubt (suspense=0.95)
Generating plot point 11/15...
  -> A Hidden Trace, A Familiar Charm (suspense=0.97)
Generating plot point 12/15...
  -> A Polymer Fingerprint, A Jud

## Phase 4: Validation + Final Narrative Assembly

In [26]:
# Functions for post-generation consistency check

def build_consistency_check_prompt(world: StoryWorld) -> str:
    payload = {
        "crime_details": {
            "crime_type": world.crime_details.crime_type if world.crime_details else "",
            "victim": world.crime_details.victim if world.crime_details else "",
            "location": world.crime_details.location if world.crime_details else "",
            "time_of_crime": world.crime_details.time_of_crime if world.crime_details else "",
            "method": world.crime_details.method if world.crime_details else "",
            "criminal_name": world.crime_details.criminal_name if world.crime_details else "",
            "criminal_motive": world.crime_details.criminal_motive if world.crime_details else "",
            "criminal_means": world.crime_details.criminal_means if world.crime_details else "",
            "backstory_summary": world.crime_details.backstory_summary if world.crime_details else "",
        },
        "characters": [
            {
                "name": c.name,
                "role": c.role,
                "description": c.description,
                "means": c.means,
                "motive": c.motive,
                "opportunity": c.opportunity,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
                "is_criminal": c.is_criminal,
                "eliminated": c.eliminated,
                "notes": c.notes,
            }
            for c in world.characters
        ],
        "clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "discovered": cl.discovered,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
                "revealed_in_plot_point": cl.revealed_in_plot_point,
            }
            for cl in world.clues
        ],
        "plot_points": [
            {
                "index": p.index,
                "title": p.title,
                "description": p.description,
                "action_taken": p.action_taken,
                "outcome": p.outcome,
                "obstacle": p.obstacle,
                "suspense_score": p.suspense_score,
                "characters_involved": p.characters_involved,
                "clues_revealed": p.clues_revealed,
                "clues_used": p.clues_used,
            }
            for p in sorted(world.plot_points, key=lambda x: x.index)
        ]
    }

    return f"""
You are checking a crime mystery story plan for contradictions and logic problems.

Review the following story materials:
{json.dumps(payload, indent=2)}

Check for:
1. Character contradictions
2. Clue contradictions
3. Impossible timeline issues
4. Plot point outcomes that conflict with later events
5. Eliminated suspects acting as if still viable later without explanation
6. Clues that appear without being revealed
7. Any mismatch between the hidden crime and the detective's reconstructed story
8. Whether the final solution points to the real criminal

Return ONLY valid JSON in exactly this format:
{{
  "is_consistent": true,
  "issues": [
    "...",
    "..."
  ],
  "summary": "Brief overall judgment."
}}
""".strip()


def run_consistency_check(world: StoryWorld) -> dict:
    raw = ask_llm(build_consistency_check_prompt(world))
    return extract_json(raw)

In [27]:
# Run consistency check on world state
consistency_report = run_consistency_check(world)
print("CONSISTENCY CHECK")
pprint(consistency_report)

CONSISTENCY CHECK
{'is_consistent': True,
 'issues': ['**Missing Clue Definitions:** Several crucial clues (Clues 9, 10, '
            '11, 12, 13, 14, 15) are revealed and extensively used within the '
            '`plot_points` section, complete with their descriptions and '
            'impacts. However, these clues are not formally defined with their '
            '`clue_id`, `description`, `location`, etc., in the top-level '
            '`clues` array of the provided story materials. While their '
            'narrative function is clear, their structured data representation '
            'is incomplete.',
            '**Clue 9 - Samuel sees Evelyn stash vial:** This clue is revealed '
            'in Plot Point 9, but its full definition is missing from the main '
            '`clues` array.',
            '**Clue 10 - EMP device:** This clue is revealed in Plot Point 10, '
            'but its full definition is missing from the main `clues` array.',
            "**Clue 11 - Xyl

In [28]:
# Functions for building narrative/prose from generated story world object
def build_narrative_assembly_prompt(world: StoryWorld) -> str:
    detective = next((c for c in world.characters if c.role == "detective"), None)

    characters_payload = [
        {
            "name": c.name,
            "role": c.role,
            "description": c.description,
            "motive": c.motive,
            "opportunity": c.opportunity,
            "alibi": c.alibi,
            "is_criminal": c.is_criminal
        }
        for c in world.characters
    ]

    clues_payload = [
        {
            "clue_id": cl.clue_id,
            "description": cl.description,
            "location": cl.location,
            "is_red_herring": cl.is_red_herring,
            "points_to": cl.points_to,
            "revealed_in_plot_point": cl.revealed_in_plot_point
        }
        for cl in world.clues
    ]

    plot_descriptions = [
        {
            "index": p.index,
            "title": p.title,
            "description": p.description,
            "action_taken": p.action_taken,
            "outcome": p.outcome,
            "obstacle": p.obstacle,
            "characters_involved": p.characters_involved,
            "clues_revealed": p.clues_revealed,
            "clues_used": p.clues_used
        }
        for p in sorted(world.plot_points, key=lambda x: x.index)
    ]

    payload = {
        "detective_name": detective.name if detective else "The detective",
        "crime_type": world.crime_details.crime_type if world.crime_details else "",
        "victim": world.crime_details.victim if world.crime_details else "",
        "location": world.crime_details.location if world.crime_details else "",
        "countdown_deadline": getattr(world, "countdown_deadline", ""),
        "countdown_consequence": getattr(world, "countdown_consequence", ""),
        "characters": characters_payload,
        "clues": clues_payload,
        "plot_points": plot_descriptions
    }

    return f"""
You are a skilled mystery novelist.

Write a complete mystery story based on the materials below.

Story materials:
{json.dumps(payload, indent=2)}

Instructions:
- Write a full, immersive narrative in the style of a mystery novel.
- Do NOT write a synopsis or summary.
- Expand the plot points into scenes with atmosphere, movement, reflection, and tension.
- Preserve the sequence of events and the underlying logic of the investigation.
- Use the characters consistently with their descriptions and roles.
- Weave clues naturally into the prose.
- Develop red herrings in a believable way.
- Maintain strong suspense throughout, with the countdown pressure growing more intense as the story progresses.
- The detective should gradually piece the truth together through observation, inference, and misdirection.
- Use polished, engaging prose and natural transitions.
- The ending should feel like a proper mystery revelation, with the detective identifying the true criminal in a satisfying way.
- A detailed and well-written story is better than a compressed one.
- Please keep it 1500-2000 words.

Literary style:
- Clear but literary prose
- Varied sentence structure
- Strong scene transitions
- Show, don’t just tell
- Dialogue should be used frequently and naturally throughout the story

Dialogue requirements:
- Include regular dialogue between characters (detective, suspects, witnesses)
- Conversations should drive the investigation forward
- Use dialogue to reveal clues, contradictions, and personality
- Avoid long stretches without dialogue
- Dialogue should feel natural, not overly formal or robotic
- Mix dialogue with action and internal thoughts

Do not:
- mention "plot point"
- mention "countdown mechanism"
- list clues explicitly
- write headings, bullet points, or outline labels

Output only the final story.
""".strip()


def assemble_final_narrative(world: StoryWorld) -> str:
    if len(world.plot_points) < 15:
        raise ValueError("Need at least 15 plot points before assembling final narrative.")

    raw = ask_llm(build_narrative_assembly_prompt(world))
    world.final_narrative = raw.strip()
    return world.final_narrative

In [29]:
# Runs final narrative assembly and prints output

final_story = assemble_final_narrative(world)
print(world.final_narrative)

The scent of expensive perfume and stale champagne clung to the Grand Ballroom of the Étoile Hotel, a ghostly reminder of the Starlight Charity Gala that had ended in tragedy. Lord Alistair Finch, a titan of real estate whose ruthlessness was legendary, lay still beneath a white sheet, his empire abruptly concluded. The initial medical assessment whispered of a heart attack, a natural end to a life lived too hard. But Detective Inspector Aris Thorne, a man whose quiet intensity belied a mind like a steel trap, felt a prickle of unease. Nothing about Lord Finch’s dramatic collapse felt ‘natural.’

“Clear the room, everyone,” Thorne’s voice, though soft, cut through the hushed murmurs of the hotel staff. He turned to a uniformed officer. “Seal off this section. I want a forensic sweep of every square inch around Finch’s table. And get me every serving staff member who approached him tonight. Start with anyone involved in beverage service.”

Marcus Chen, the hotel’s perpetually stressed G

## Phase II Architecture & Code Map

| Architecture component                        | Cell / function                              |
|-----------------------------------------------|----------------------------------------------|
| Story Generator (Phase I)                     | All cells above this section                 |
| Plot Graph (DAG of nodes w/ pre/effect preds) | Step 1 — `PlotNode`, `extract_plot_graph`    |
| Game World (locations, items, NPC placement) | Step 2 — `GameWorld`, `generate_game_world`  |
| World State (runtime, cheap to clone)        | Step 3 — `WorldState`, `apply_effect`        |
| NL Action Parser                              | Step 4 — `parse_action`                      |
| Game Engine (trigger & validation)           | Step 5 — `GameEngine.step`                   |
| Drama Manager search                          | Step 6 — `candidate_interventions`, rollouts, `score_state`, `drama_manager_decide` |
| Intervention rendering                        | Step 7 — `render_intervention`               |
| Interactive player loop                       | Step 8 — `play`                              |
| Evaluation harness + walkthroughs            | Step 9 — `run_eval`, walkth


# Phase II: Search-Based Drama Manager

Phase 1 above produces a finished, linear story. Phase II reuses everything in `world` (crime, characters, clues, plot points, countdown) and turns it into a **playable, text-based interactive mystery** governed by a search-based drama manager (DM).

Pipeline (matches the Phase II project specification):
1. **Plot Graph** — convert the 15 linear plot points into an abstract DAG of nodes with preconditions and effects.
2. **World Generation** — generate locations, items, and NPC placements that support every node's preconditions.
3. **World State** — runtime state that is cheap to clone for DM rollouts.
4. **Action Parser** — natural-language input → structured `(verb, target, indirect)` action.
5. **Game Engine** — validates the action, updates state, fires any plot-graph nodes whose preconditions are now satisfied (constituent / exceptional / neutral classification).
6. **Drama Manager** — sampling-based rollouts over candidate interventions (`hint`, `temp_deny`, `cause`, `glow`, `noop`); picks the move that maximises a utility combining solvability, pacing, suspense, and (negative) manipulation.
7. **Intervention Rendering** — translate the chosen DM move into diegetic prose.
8. **Game Loop** — interactive REPL.
9. **Evaluation Harness** — non-interactive auto-play with metrics from the project plan.

## Step 1 — Plot Graph

Each linear plot point becomes a `PlotNode` with explicit `preconditions` and `effects`. Predicates use a tiny vocabulary so the engine can check them deterministically:

- `["clue_discovered", <id>]`
- `["interviewed", <name>]`
- `["at_location", <name>]`
- `["suspect_eliminated", <name>]`
- `["item_obtained", <name>]`
- `["flag", <name>]`

In [30]:
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple, Callable
import copy
import random


@dataclass
class PlotNode:
    """One node in the abstract plot graph."""
    id: int
    title: str
    description: str
    preconditions: List[List[Any]] = field(default_factory=list)
    effects: List[List[Any]] = field(default_factory=list)
    is_terminal: bool = False  # True if this node represents the case being solved


def build_plot_graph_prompt(world: StoryWorld) -> str:
    plot_payload = [
        {
            "index": p.index,
            "title": p.title,
            "description": p.description,
            "action_taken": p.action_taken,
            "outcome": p.outcome,
            "characters_involved": p.characters_involved,
            "clues_revealed": p.clues_revealed,
            "clues_used": p.clues_used,
        }
        for p in sorted(world.plot_points, key=lambda x: x.index)
    ]
    detective = next((c.name for c in world.characters if c.role == "detective"), "Detective")
    suspects = [c.name for c in world.characters if c.role in ("suspect", "criminal")]
    criminal = next((c.name for c in world.characters if c.is_criminal), "")

    return f"""
You are converting a linear mystery story into an INTERACTIVE plot graph for a text game.

Each plot point becomes a node with:
- preconditions: facts that must be true before the node can fire
- effects: facts that become true once it fires

Use ONLY these predicate types:
- ["clue_discovered", <clue_id:int>]
- ["interviewed", <character_name:str>]
- ["at_location", <location_name:str>]
- ["item_obtained", <item_name:str>]
- ["flag", <flag_name:str>]
- ["accuse", <character_name:str>]    # ONLY allowed as a precondition of the terminal node

Detective: {detective}
Suspects: {suspects}
Criminal (true answer): {criminal}

Original linear plot points:
{json.dumps(plot_payload, indent=2)}

Return ONLY valid JSON:
{{
  "nodes": [
    {{
      "id": 1,
      "title": "...",
      "description": "...",
      "preconditions": [["at_location", "Library"], ["clue_discovered", 1]],
      "effects": [["flag", "method_understood"]],
      "is_terminal": false
    }}
  ]
}}

HARD RULES (the game does not work if you violate these):
1. Node ids equal the original plot-point indices (1..N).
2. EFFECTS MUST NOT INCLUDE player-action predicates: `clue_discovered`, `interviewed`,
   `at_location`, or `accuse`. Clues are discovered by `examine`, interviews are set
   by `interview`, and location is set by `move`. Use `flag` predicates for narrative
   state changes (e.g., ["flag", "lab_results_in"]).
3. PRECONDITIONS for every non-starter, non-terminal node MUST include at least one
   player-driven predicate from {{`at_location`, `interviewed`, `item_obtained`}}, in
   addition to any chain conditions like `clue_discovered` or `flag`.
4. Mark exactly ONE final node as is_terminal=true. Its preconditions MUST include
   ["accuse", "{criminal}"] AND require enough discovered clues to justify the accusation.
5. Up to 2 starter nodes (those that can fire at game start) may have empty preconditions
   — these establish the opening scene. All other nodes must have at least one predicate.
6. Stay grounded in the existing story; do not invent new clues or characters.
""".strip()


def extract_plot_graph(world: StoryWorld) -> List[PlotNode]:
    raw = ask_llm(build_plot_graph_prompt(world))
    data = extract_json(raw)
    nodes = []
    for n in data["nodes"]:
        # Strip player-action predicates from effects (defensive). These should only
        # be set by the player actually doing the action in the game engine.
        blocked_effects = {"clue_discovered", "interviewed", "at_location", "accuse"}
        effs = [list(e) for e in n.get("effects", []) if e and e[0] not in blocked_effects]
        nodes.append(PlotNode(
            id=int(n["id"]),
            title=n["title"],
            description=n["description"],
            preconditions=[list(p) for p in n.get("preconditions", [])],
            effects=effs,
            is_terminal=bool(n.get("is_terminal", False)),
        ))
    if not any(n.is_terminal for n in nodes) and nodes:
        nodes[-1].is_terminal = True
    return nodes



In [31]:
# Runner: extract the plot graph
world.plot_graph = extract_plot_graph(world)

print(f"Plot graph: {len(world.plot_graph)} nodes")
for n in world.plot_graph:
    flag = " [TERMINAL]" if n.is_terminal else ""
    print(f"[{n.id}] {n.title}{flag}")
    print(f"    pre: {n.preconditions}")
    print(f"    eff: {n.effects}")


Plot graph: 15 nodes
[1] A Flicker in the Dark, a Peculiar Ice Cube
    pre: []
    eff: [['flag', 'initial_investigation_complete']]
[2] A Chilling Find in Cold Storage
    pre: [['clue_discovered', 8], ['at_location', 'Catering Kitchen'], ['interviewed', 'Evelyn Reed']]
    eff: [['flag', 'ice_mold_discovered']]
[3] A Sedative's Whisper in the Cold
    pre: [['clue_discovered', 1], ['clue_discovered', 7], ['clue_discovered', 8], ['interviewed', 'Marcus Chen'], ['at_location', "Evelyn Reed's Office"]]
    eff: [['flag', 'xylazine_syringe_found']]
[4] The Judge's Tightened Grip
    pre: [['clue_discovered', 1], ['clue_discovered', 2], ['clue_discovered', 7], ['clue_discovered', 8], ['interviewed', 'Temporary Kitchen Assistant']]
    eff: [['flag', 'embalming_deadline_set'], ['flag', 'judge_demands_direct_evidence']]
[5] Encrypted Secrets, Judicial Delays
    pre: [['clue_discovered', 1], ['clue_discovered', 2], ['clue_discovered', 4], ['clue_discovered', 7], ['clue_discovered', 8], ['i

## Step 2 — World Generation

Generate the explorable map: locations with adjacency, items (one per clue), and starting positions for the detective and every NPC.

In [32]:
@dataclass
class Location:
    name: str
    description: str
    neighbors: List[str] = field(default_factory=list)


@dataclass
class Item:
    name: str
    description: str
    location: str
    related_clue_id: int = -1


@dataclass
class GameWorld:
    locations: List[Location] = field(default_factory=list)
    items: List[Item] = field(default_factory=list)
    npc_locations: Dict[str, str] = field(default_factory=dict)
    starting_location: str = ""

    def get_location(self, name: str) -> Optional[Location]:
        for loc in self.locations:
            if loc.name.lower() == name.lower():
                return loc
        return None


def build_world_gen_prompt(world: StoryWorld) -> str:
    crime = world.crime_details
    char_payload = [
        {"name": c.name, "role": c.role, "description": c.description}
        for c in world.characters
    ]
    clue_payload = [
        {"clue_id": cl.clue_id, "description": cl.description, "location": cl.location}
        for cl in world.clues
    ]
    return f"""
You are building a clear, playable text-game map for an interactive crime mystery.

Crime: {crime.crime_type} of {crime.victim} at {crime.location} ({crime.time_of_crime}).

Characters:
{json.dumps(char_payload, indent=2)}

Clues (the 'location' field is the original story description, which you should map to one of your generated locations):
{json.dumps(clue_payload, indent=2)}

Generate:
- 6 to 10 distinct, named locations covering the story setting.
- An undirected adjacency graph (commonsense neighbours; map must be connected).
- One starting location for the detective (typically arrival point or main hall).
- One item per clue; each item is the physical object the player examines to discover the clue, placed in a sensible location.
- A starting location for every non-detective character (suspect / criminal / witness).

Return ONLY valid JSON:
{{
  "locations": [
    {{"name": "...", "description": "...", "neighbors": ["...", "..."]}}
  ],
  "items": [
    {{"name": "...", "description": "...", "location": "...", "related_clue_id": 1}}
  ],
  "npc_locations": {{"Character Name": "Location Name"}},
  "starting_location": "..."
}}

Rules:
- Adjacency must be (mostly) symmetric. If A lists B, B should list A.
- Every location must be reachable from starting_location.
- Every clue id (1..N) must have exactly one corresponding item.
- npc_locations values must reference an existing location name.
- Names must be unique. Keep them short (1-3 words).
- Keep every location description to ONE plain sentence under 22 words.
- Keep every item description to ONE plain sentence under 18 words that says what the player notices.
- Avoid ornate, metaphor-heavy prose. The player should immediately understand what is in the room and why it matters.
""".strip()


def generate_game_world(world: StoryWorld) -> GameWorld:
    raw = ask_llm(build_world_gen_prompt(world))
    data = extract_json(raw)
    gw = GameWorld(
        locations=[
            Location(name=l["name"], description=l["description"], neighbors=list(l.get("neighbors", [])))
            for l in data["locations"]
        ],
        items=[
            Item(
                name=i["name"],
                description=i["description"],
                location=i["location"],
                related_clue_id=int(i.get("related_clue_id", -1)),
            )
            for i in data["items"]
        ],
        npc_locations=dict(data.get("npc_locations", {})),
        starting_location=data.get("starting_location", ""),
    )
    # Symmetrize adjacency
    by_name = {l.name: l for l in gw.locations}
    for loc in gw.locations:
        for n in list(loc.neighbors):
            if n in by_name and loc.name not in by_name[n].neighbors:
                by_name[n].neighbors.append(loc.name)
    return gw


In [33]:
# Runner: generate the game world
world.game_world = generate_game_world(world)

gw = world.game_world
print(f"Locations: {len(gw.locations)} | Items: {len(gw.items)} | NPCs placed: {len(gw.npc_locations)}")
print(f"Start: {gw.starting_location}\n")
print("Map:")
for loc in gw.locations:
    print(f"  - {loc.name}: {loc.neighbors}")
print("\nItems:")
for it in gw.items:
    print(f"  - {it.name} (clue {it.related_clue_id}) @ {it.location}")
print("\nNPCs:")
for name, loc in gw.npc_locations.items():
    print(f"  - {name} @ {loc}")


Locations: 8 | Items: 8 | NPCs placed: 6
Start: Grand Ballroom

Map:
  - Grand Ballroom: ['Hotel Lobby', 'Service Hallway']
  - Hotel Lobby: ['Grand Ballroom', "Members' Lounge", 'Service Hallway', 'Hotel Management Office']
  - Catering Kitchen: ['Service Hallway', 'Secluded Pantry', "Head Caterer's Office"]
  - Head Caterer's Office: ['Catering Kitchen']
  - Secluded Pantry: ['Catering Kitchen']
  - Members' Lounge: ['Hotel Lobby', 'Service Hallway']
  - Service Hallway: ['Grand Ballroom', 'Hotel Lobby', 'Catering Kitchen', "Members' Lounge", 'Hotel Management Office']
  - Hotel Management Office: ['Hotel Lobby', 'Service Hallway']

Items:
  - Silicone Ice Mold (clue 1) @ Catering Kitchen
  - Disposable Syringe (clue 2) @ Head Caterer's Office
  - Evelyn's Tablet (clue 3) @ Head Caterer's Office
  - Staff Interview Transcripts (clue 4) @ Grand Ballroom
  - Wait Staff Statement (clue 5) @ Grand Ballroom
  - Bartender's Testimony (clue 6) @ Members' Lounge
  - Electrical Log Book (clue

## Step 3 — World State

`WorldState` is the runtime state for one play-through.

In [34]:
@dataclass
class WorldState:
    """Runtime state for a single play-through. Cheap to clone for DM rollouts."""
    player_location: str = ""
    inventory: List[str] = field(default_factory=list)
    discovered_clues: List[int] = field(default_factory=list)
    interviewed: List[str] = field(default_factory=list)
    eliminated_suspects: List[str] = field(default_factory=list)
    flags: Dict[str, bool] = field(default_factory=dict)
    executed_nodes: List[int] = field(default_factory=list)
    perm_denied_nodes: List[int] = field(default_factory=list)
    temp_denied_nodes: Dict[int, int] = field(default_factory=dict)  # node_id -> turn it expires
    turn: int = 0
    intervention_count: int = 0
    case_solved: bool = False
    last_classification: str = ""
    accused: str = ""  # set when player accuses a suspect; used for terminal node firing


def predicate_satisfied(state: WorldState, pred: List[Any]) -> bool:
    if not pred:
        return True
    kind = pred[0]
    arg = pred[1] if len(pred) > 1 else None
    if kind == "clue_discovered":
        return int(arg) in state.discovered_clues
    if kind == "interviewed":
        return arg in state.interviewed
    if kind == "at_location":
        return state.player_location == arg
    if kind == "suspect_eliminated":
        return arg in state.eliminated_suspects
    if kind == "item_obtained":
        return arg in state.inventory
    if kind == "flag":
        return bool(state.flags.get(arg, False))
    if kind == "accuse":
        return state.accused == arg
    return False


def apply_effect(state: WorldState, eff: List[Any]) -> None:
    if not eff:
        return
    kind = eff[0]
    arg = eff[1] if len(eff) > 1 else None
    if kind == "clue_discovered":
        cid = int(arg)
        if cid not in state.discovered_clues:
            state.discovered_clues.append(cid)
    elif kind == "interviewed" and arg not in state.interviewed:
        state.interviewed.append(arg)
    elif kind == "suspect_eliminated" and arg not in state.eliminated_suspects:
        state.eliminated_suspects.append(arg)
    elif kind == "item_obtained" and arg not in state.inventory:
        state.inventory.append(arg)
    elif kind == "flag":
        state.flags[arg] = True
    # accuse is intentionally NOT applied via effects — it is set on the state by
    # the engine when the player issues an accuse action.


def apply_plot_effect(state: WorldState, eff: List[Any]) -> None:
    """Apply effects generated by plot nodes or DM interventions.

    Player-action predicates should only be set by the player actually doing the
    action. Otherwise a plot node can accidentally mark every witness as already
    interviewed, which blocks later interviews.
    """
    if not eff:
        return
    if eff[0] in ("clue_discovered", "interviewed", "at_location", "accuse"):
        return
    apply_effect(state, eff)


def initialize_state(world: StoryWorld) -> WorldState:
    """Create the starting state and fire any starter nodes (empty preconditions,
    non-terminal) ONCE without cascading. The cascade is intentionally avoided
    here — the engine\'s delta-based trigger drives all later firings."""
    state = WorldState(player_location=world.game_world.starting_location)
    for node in world.plot_graph:
        if node.is_terminal:
            continue
        if not node.preconditions and node.id not in state.executed_nodes:
            for eff in node.effects:
                apply_plot_effect(state, eff)
            state.executed_nodes.append(node.id)
    return state


def clone_state(state: WorldState) -> WorldState:
    return copy.deepcopy(state)



## Step 4 — Action Parser

`parse_action` turns the player's free-text input into a structured `Action` using the LLM.

In [35]:
@dataclass
class Action:
    verb: str = "unknown"      # move | examine | interview | ask_about | use_item | accuse | look | unknown
    target: str = ""
    indirect: str = ""
    raw: str = ""


import difflib


VALID_VERBS = {"move", "examine", "interview", "ask_about", "use_item", "accuse", "look", "unknown"}
STOP_WORDS = {
    "a", "an", "and", "at", "go", "in", "into", "look", "move", "of", "on", "please",
    "read", "search", "see", "the", "to", "with", "examine", "inspect", "check", "talk",
    "speak", "interview", "question", "ask", "about", "accuse", "use", "open", "room"
}
EXAMINE_PREFIXES = ["look at", "look in", "look inside", "examine", "inspect", "search", "check", "read", "open"]
MOVE_PREFIXES = ["go to", "move to", "walk to", "enter", "head to", "travel to", "go", "move", "walk"]
TALK_PREFIXES = ["talk to", "speak to", "interview", "question", "ask"]
ACCUSE_PREFIXES = ["accuse", "arrest"]
USE_PREFIXES = ["use"]


def _norm(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()


def _tokens(text: str) -> List[str]:
    toks = []
    for t in _norm(text).split():
        if not t or t in STOP_WORDS:
            continue
        if len(t) > 3 and t.endswith("s"):
            t = t[:-1]
        toks.append(t)
    return toks


def _best_match(text: str, choices: List[str], *, min_score: float = 0.25) -> str:
    """Return the canonical choice matching a partial player phrase, or ''."""
    if not text or not choices:
        return ""
    nt = _norm(text)
    if not nt:
        return ""
    normalized = {c: _norm(c) for c in choices}
    for choice, nc in normalized.items():
        if nt == nc:
            return choice
    for choice, nc in normalized.items():
        if nt in nc or nc in nt:
            return choice

    wanted = set(_tokens(text))
    best_choice = ""
    best_score = 0.0
    for choice in choices:
        have = set(_tokens(choice))
        token_score = (len(wanted & have) / max(len(wanted), len(have))) if wanted and have else 0.0
        seq_score = difflib.SequenceMatcher(None, nt, normalized[choice]).ratio() * 0.75
        score = max(token_score, seq_score)
        if score > best_score:
            best_choice = choice
            best_score = score
    return best_choice if best_score >= min_score else ""


def visible_items(world: StoryWorld, state: WorldState) -> List[str]:
    return [it.name for it in world.game_world.items if it.location == state.player_location]


def visible_npcs(world: StoryWorld, state: WorldState) -> List[str]:
    return [name for name, loc in world.game_world.npc_locations.items() if loc == state.player_location]


def active_suspect_names(world: StoryWorld, state: WorldState) -> List[str]:
    return [c.name for c in world.characters
            if c.role in ("suspect", "criminal") and c.name not in state.eliminated_suspects]


def state_summary_for_parser(world: StoryWorld, state: WorldState) -> str:
    here = world.game_world.get_location(state.player_location)
    return json.dumps({
        "current_location": state.player_location,
        "neighbors": here.neighbors if here else [],
        "items_here": visible_items(world, state),
        "npcs_here": visible_npcs(world, state),
        "inventory": state.inventory,
        "interviewed_so_far": state.interviewed,
        "discovered_clue_ids": state.discovered_clues,
        "remaining_active_suspects": active_suspect_names(world, state),
    }, indent=2)


def _starts_with_any(text: str, prefixes: List[str]) -> bool:
    lowered = text.lower().strip()
    return any(lowered == p or lowered.startswith(p + " ") for p in prefixes)


def _command_tail(text: str, prefixes: List[str]) -> str:
    lowered = text.lower().strip()
    for prefix in sorted(prefixes, key=len, reverse=True):
        if lowered == prefix:
            return ""
        if lowered.startswith(prefix + " "):
            return text[len(prefix):].strip(" :.-")
    return text.strip(" :.-")


def local_parse_action(world: StoryWorld, state: WorldState, nl_text: str) -> Action:
    text = nl_text.strip()
    lowered = text.lower().strip()
    here = world.game_world.get_location(state.player_location)
    neighbors = here.neighbors if here else []
    items = visible_items(world, state)
    npcs = visible_npcs(world, state)
    suspects = active_suspect_names(world, state)

    if lowered in ("look", "l", "look around", "where am i", "where", "help", "commands", "inventory"):
        return Action(verb="look", raw=nl_text)

    if _starts_with_any(text, EXAMINE_PREFIXES):
        target_text = _command_tail(text, EXAMINE_PREFIXES)
        if not target_text or _best_match(target_text, [state.player_location], min_score=0.35):
            return Action(verb="look", raw=nl_text)
        target = _best_match(target_text, items) or _best_match(text, items) or target_text
        return Action(verb="examine", target=target, raw=nl_text)

    if _starts_with_any(text, MOVE_PREFIXES):
        target_text = _command_tail(text, MOVE_PREFIXES)
        target = _best_match(target_text, neighbors) or _best_match(text, neighbors) or target_text
        return Action(verb="move", target=target, raw=nl_text)

    if _starts_with_any(text, TALK_PREFIXES):
        target_text = _command_tail(text, TALK_PREFIXES)
        topic = ""
        if " about " in target_text.lower():
            before, topic = re.split(r"\s+about\s+", target_text, maxsplit=1, flags=re.IGNORECASE)
            target_text = before.strip()
            topic = topic.strip()
        target = _best_match(target_text, npcs) or _best_match(text, npcs) or target_text
        if topic:
            return Action(verb="ask_about", target=target, indirect=topic, raw=nl_text)
        return Action(verb="interview", target=target, raw=nl_text)

    if _starts_with_any(text, ACCUSE_PREFIXES):
        target_text = _command_tail(text, ACCUSE_PREFIXES)
        target = _best_match(target_text, suspects) or _best_match(text, suspects) or target_text
        return Action(verb="accuse", target=target, raw=nl_text)

    if _starts_with_any(text, USE_PREFIXES):
        target_text = _command_tail(text, USE_PREFIXES)
        target, indirect = target_text, ""
        if " on " in target_text.lower():
            target, indirect = re.split(r"\s+on\s+", target_text, maxsplit=1, flags=re.IGNORECASE)
        return Action(verb="use_item", target=target.strip(), indirect=indirect.strip(), raw=nl_text)

    for choices, verb in ((items, "examine"), (neighbors, "move"), (npcs, "interview"), (suspects, "accuse")):
        target = _best_match(text, choices, min_score=0.35)
        if target:
            return Action(verb=verb, target=target, raw=nl_text)

    return Action(verb="unknown", raw=nl_text)


def canonicalize_action(world: StoryWorld, state: WorldState, action: Action) -> Action:
    action.verb = action.verb if action.verb in VALID_VERBS else "unknown"
    if action.verb == "move":
        here = world.game_world.get_location(state.player_location)
        action.target = _best_match(action.target or action.raw, here.neighbors if here else []) or action.target
    elif action.verb == "examine":
        if _best_match(action.target or action.raw, [state.player_location], min_score=0.35):
            action.verb = "look"
            action.target = ""
        else:
            action.target = _best_match(action.target or action.raw, visible_items(world, state)) or action.target
    elif action.verb in ("interview", "ask_about"):
        action.target = _best_match(action.target or action.raw, visible_npcs(world, state)) or action.target
    elif action.verb == "accuse":
        action.target = _best_match(action.target or action.raw, active_suspect_names(world, state)) or action.target
    elif action.verb == "look" and action.target:
        item = _best_match(action.target, visible_items(world, state))
        if item:
            action.verb = "examine"
            action.target = item
    return action


def parse_action(world: StoryWorld, state: WorldState, nl_text: str) -> Action:
    local = local_parse_action(world, state, nl_text)
    if local.verb != "unknown":
        return canonicalize_action(world, state, local)

    prompt = f"""
You are an action parser for a text mystery game. Convert the player's sentence into ONE action.

Use one of these verbs only:
- "move"      target must exactly match one neighboring location
- "examine"   target must exactly match one visible item
- "interview" target must exactly match one visible NPC
- "ask_about" target must exactly match one visible NPC; indirect is the topic
- "use_item"  target must exactly match one inventory item; indirect is optional
- "accuse"    target must exactly match one active suspect; this is the final answer
- "look"      use when the player wants to survey the room
- "unknown"   use when no legal action matches

Important:
- If the player says "look at", "inspect", "search", "read", "check", or "open" a visible object, return "examine".
- Prefer the closest exact name from the current state instead of inventing a new target.
- Keep JSON values short. Do not explain the action.

Current state:
{state_summary_for_parser(world, state)}

Player input: {nl_text!r}

Return ONLY valid JSON: {{"verb": "...", "target": "...", "indirect": ""}}
""".strip()
    try:
        raw = ask_llm(prompt)
        data = extract_json(raw)
        parsed = Action(
            verb=data.get("verb", "unknown"),
            target=data.get("target", "") or "",
            indirect=data.get("indirect", "") or "",
            raw=nl_text,
        )
        return canonicalize_action(world, state, parsed)
    except Exception:
        return Action(verb="unknown", raw=nl_text)


## Step 5 — Game Engine (trigger & validation)

`GameEngine.step` is the heart of the loop:

1. Validate the action against the current state.
2. Apply the action's direct effects (move location, mark a clue discovered when the right item is examined, mark an NPC interviewed, etc.).
3. Walk the plot graph; any pending node whose preconditions are now satisfied (and which is not denied by the DM) fires and applies its own effects.
4. Classify the action: `constituent` if it advanced a node, `case_solved` / `false_accusation` for accusations, otherwise `neutral` or `invalid`.

In [36]:
class GameEngine:
    def __init__(self, world: StoryWorld):
        self.world = world

    def _is_legal(self, state: WorldState, action: Action) -> Tuple[bool, str]:
        gw = self.world.game_world
        if action.verb == "move":
            here = gw.get_location(state.player_location)
            if not here or action.target not in here.neighbors:
                exits = ", ".join(here.neighbors) if here else "none"
                return False, f"You can't go to {action.target!r} from here. Exits: {exits}."
        elif action.verb == "examine":
            it = next((i for i in gw.items if i.name == action.target and i.location == state.player_location), None)
            if not it:
                items = visible_items(self.world, state)
                if items:
                    return False, f"You do not see {action.target!r} here. Try examining: {', '.join(items)}."
                return False, "There is nothing obvious here to examine. Try 'look' or move to another room."
        elif action.verb == "interview":
            if gw.npc_locations.get(action.target) != state.player_location:
                npcs = visible_npcs(self.world, state)
                if npcs:
                    return False, f"{action.target} is not in this room. Present: {', '.join(npcs)}."
                return False, "There is nobody here to interview. Try 'look' or move to another room."
        elif action.verb == "ask_about":
            if gw.npc_locations.get(action.target) != state.player_location:
                return False, f"{action.target} is not here to question."
        elif action.verb == "use_item":
            if action.target not in state.inventory:
                if state.inventory:
                    return False, f"You are not carrying {action.target!r}. Inventory: {', '.join(state.inventory)}."
                return False, "You are not carrying anything useful yet. Examine evidence first."
        elif action.verb == "accuse":
            names = [c.name for c in self.world.characters if c.role in ("suspect", "criminal")]
            if action.target not in names:
                return False, f"{action.target} is not a known suspect. Known suspects: {', '.join(names)}."
        elif action.verb == "unknown":
            return False, "I didn't understand that. Try 'look', 'examine <item>', 'move to <room>', 'interview <name>', or 'accuse <name>'."
        return True, ""

    def _apply_action_effects(self, state: WorldState, action: Action) -> None:
        gw = self.world.game_world
        if action.verb == "move":
            state.player_location = action.target
        elif action.verb == "examine":
            it = next((i for i in gw.items if i.name == action.target and i.location == state.player_location), None)
            if it and it.related_clue_id > 0:
                apply_effect(state, ["clue_discovered", it.related_clue_id])
        elif action.verb == "interview":
            apply_effect(state, ["interviewed", action.target])

    def _trigger_nodes(self, state: WorldState, prev: WorldState) -> List[int]:
        """Delta-based: fire AT MOST ONE node per turn, and only when at least
        one precondition just transitioned from unsatisfied to satisfied. Never
        auto-fire is_terminal nodes — those only fire on a successful accuse."""
        for node in self.world.plot_graph:
            if node.is_terminal:
                continue
            if node.id in state.executed_nodes or node.id in state.perm_denied_nodes:
                continue
            if node.id in state.temp_denied_nodes and state.turn < state.temp_denied_nodes[node.id]:
                continue
            if not all(predicate_satisfied(state, p) for p in node.preconditions):
                continue
            if node.preconditions:
                transitioned = any(
                    predicate_satisfied(state, p) and not predicate_satisfied(prev, p)
                    for p in node.preconditions
                )
                if not transitioned:
                    continue
            else:
                # Empty-precondition starter nodes were fired in initialize_state;
                # they should not re-fire mid-game.
                continue
            for eff in node.effects:
                apply_plot_effect(state, eff)
            state.executed_nodes.append(node.id)
            return [node.id]
        return []

    def step(self, state: WorldState, action: Action) -> Dict[str, Any]:
        action = canonicalize_action(self.world, state, action)
        state.turn += 1
        legal, msg = self._is_legal(state, action)
        if not legal:
            state.last_classification = "invalid"
            return {"ok": False, "message": msg, "fired_nodes": [], "classification": "invalid"}

        # Accusation is a terminal verb handled separately.
        if action.verb == "accuse":
            criminal = next((c.name for c in self.world.characters if c.is_criminal), "")
            min_clues_needed = max(3, len(self.world.clues) // 2)
            state.accused = action.target
            if action.target == criminal and len(state.discovered_clues) >= min_clues_needed:
                # Fire the terminal node if one exists.
                term = next((n for n in self.world.plot_graph if n.is_terminal), None)
                if term and term.id not in state.executed_nodes:
                    for eff in term.effects:
                        apply_plot_effect(state, eff)
                    state.executed_nodes.append(term.id)
                state.case_solved = True
                state.last_classification = "case_solved"
                return {"ok": True, "message": "", "fired_nodes": [term.id] if term else [],
                        "classification": "case_solved", "case_solved": True}
            state.last_classification = "false_accusation"
            return {"ok": True, "message": "", "fired_nodes": [],
                    "classification": "false_accusation", "case_solved": False}

        # Snapshot BEFORE applying the action so the trigger can compute deltas.
        prev = clone_state(state)
        self._apply_action_effects(state, action)
        fired = self._trigger_nodes(state, prev)
        classification = "constituent" if fired else "neutral"
        state.last_classification = classification
        return {
            "ok": True,
            "message": "",
            "fired_nodes": fired,
            "classification": classification,
            "case_solved": state.case_solved,
        }




## Step 6 — Drama Manager (search & scoring)

The DM enumerates a small set of interventions, simulates a few short rollouts forward, scores the resulting states, and picks the argmax.

**Scoring** combines four criteria:

- `+ solvability`  — a path to the terminal node still exists
- `+ pacing`       — node-completion progress tracks turn count
- `+ suspense`     — reward clue discovery
- `− manipulation` — soft cap on number of interventions used

In [42]:
@dataclass
class Intervention:
    kind: str           # noop | hint | glow | temp_deny | perm_deny | cause
    target: Any = None
    ttl_turns: int = 0

    def short(self) -> str:
        if self.kind == "noop":
            return "noop"
        if self.kind == "temp_deny":
            return f"temp_deny(node={self.target}, ttl={self.ttl_turns})"
        return f"{self.kind}({self.target})"


@dataclass
class DMTrace:
    """Per-turn record of what the drama manager thought about."""
    turn: int
    fired_nodes: List[int]
    classification: str
    candidates: List[Dict[str, Any]] = field(default_factory=list)  # [{move, score}]
    chosen: str = ""
    chosen_score: float = 0.0
    reason: str = ""
    state_snapshot: Dict[str, Any] = field(default_factory=dict)

    def render(self) -> str:
        top = sorted(self.candidates, key=lambda x: -x["score"])[:5]
        lines = [
            f"[DM turn {self.turn}] classification={self.classification} fired={self.fired_nodes}",
            f"  state: clues={self.state_snapshot.get('clues',[])} "
            f"executed={self.state_snapshot.get('executed',[])} "
            f"interventions={self.state_snapshot.get('interventions',0)}",
            f"  considered {len(self.candidates)} moves; top:",
        ]
        for c in top:
            mark = " <-- chosen" if c["move"] == self.chosen else ""
            lines.append(f"    {c['score']:+.3f}  {c['move']}{mark}")
        if self.reason:
            lines.append(f"  reason: {self.reason}")
        return "\n".join(lines)


def candidate_interventions(world: StoryWorld, state: WorldState) -> List[Intervention]:
    cands: List[Intervention] = [Intervention(kind="noop")]

    # Hints toward undiscovered clues
    for cl in world.clues:
        if cl.clue_id not in state.discovered_clues:
            cands.append(Intervention(kind="hint", target=cl.clue_id))

    # Glow: highlight an item in the player's current room with an undiscovered clue
    for it in world.game_world.items:
        if (it.related_clue_id > 0
                and it.related_clue_id not in state.discovered_clues
                and it.location == state.player_location):
            cands.append(Intervention(kind="glow", target=it.name))

    # temp_deny: any currently fire-able node — keeps the case from resolving too fast
    for node in world.plot_graph:
        if node.id in state.executed_nodes or node.id in state.perm_denied_nodes:
            continue
        if all(predicate_satisfied(state, p) for p in node.preconditions):
            if not node.is_terminal:
                cands.append(Intervention(kind="temp_deny", target=node.id, ttl_turns=3))

    # cause: nudge a near-ready node forward (<= 1 missing precondition)
    for node in world.plot_graph:
        if node.id in state.executed_nodes or node.id in state.perm_denied_nodes:
            continue
        missing = [p for p in node.preconditions if not predicate_satisfied(state, p)]
        if 0 < len(missing) <= 1 and not node.is_terminal:
            cands.append(Intervention(kind="cause", target=node.id))

    return cands


def apply_intervention(world: StoryWorld, state: WorldState, iv: Intervention) -> None:
    if iv.kind == "noop":
        return
    state.intervention_count += 1
    if iv.kind == "temp_deny":
        state.temp_denied_nodes[int(iv.target)] = state.turn + iv.ttl_turns
    elif iv.kind == "perm_deny":
        if int(iv.target) not in state.perm_denied_nodes:
            state.perm_denied_nodes.append(int(iv.target))
    elif iv.kind == "cause":
        node = next((n for n in world.plot_graph if n.id == int(iv.target)), None)
        if node and node.id not in state.executed_nodes:
            for eff in node.effects:
                apply_plot_effect(state, eff)
            state.executed_nodes.append(node.id)
            if node.is_terminal:
                state.case_solved = True
    # hint / glow are pure narration


def solvable(world: StoryWorld, state: WorldState) -> bool:
    terminals = [n for n in world.plot_graph if n.is_terminal]
    if not terminals:
        return True
    return any(t.id not in state.perm_denied_nodes for t in terminals)


def score_state(world: StoryWorld, state: WorldState, intervention_budget: int = 8) -> float:
    score = 0.0
    score += 1.0 if solvable(world, state) else -2.0
    progress = len(state.executed_nodes) / max(1, len(world.plot_graph))
    expected = min(1.0, state.turn / 20.0)
    score += 1.0 - abs(progress - expected)            # pacing
    score += 0.1 * len(state.discovered_clues)         # suspense / clue flow
    score -= 0.25 * max(0, state.intervention_count - intervention_budget)  # anti-manipulation
    if state.case_solved:
        score += 2.0
    return score


def player_model_action(world: StoryWorld, state: WorldState) -> Action:
    """Heuristic player used inside DM rollouts and as a baseline auto-player."""
    gw = world.game_world
    for it in gw.items:
        if it.location == state.player_location and it.related_clue_id not in state.discovered_clues:
            return Action(verb="examine", target=it.name)
    for name, loc in gw.npc_locations.items():
        if loc == state.player_location and name not in state.interviewed:
            return Action(verb="interview", target=name)
    min_clues_needed = max(3, len(world.clues) // 2)
    if len(state.discovered_clues) >= min_clues_needed:
        criminal = next((c.name for c in world.characters if c.is_criminal), None)
        if criminal:
            return Action(verb="accuse", target=criminal)
    here = gw.get_location(state.player_location)
    if here and here.neighbors:
        return Action(verb="move", target=random.choice(here.neighbors))
    return Action(verb="look")


def rollout(world: StoryWorld, engine, start: WorldState, depth: int = 5) -> WorldState:
    state = clone_state(start)
    for _ in range(depth):
        if state.case_solved:
            break
        action = player_model_action(world, state)
        engine.step(state, action)
    return state


def explain_choice(world: StoryWorld, state: WorldState, iv: Intervention) -> str:
    progress = len(state.executed_nodes) / max(1, len(world.plot_graph))
    expected = min(1.0, state.turn / 20.0)
    pacing = "ahead of pace" if progress > expected else ("behind pace" if progress < expected else "on pace")
    if iv.kind == "noop":
        return f"player has agency; story is {pacing}"
    if iv.kind == "hint":
        return f"player has only {len(state.discovered_clues)}/{len(world.clues)} clues; nudge clue {iv.target}"
    if iv.kind == "glow":
        return f"item with an unrevealed clue is in the player's room; surface it"
    if iv.kind == "temp_deny":
        return f"throttling node {iv.target} for {iv.ttl_turns} turns ({pacing})"
    if iv.kind == "cause":
        return f"node {iv.target} is one precondition away; fire it to keep momentum"
    return ""


def drama_manager_decide(
    world: StoryWorld,
    engine,
    state: WorldState,
    num_samples: int = 3,
    depth: int = 4,
    return_trace: bool = False,
):
    cands = candidate_interventions(world, state)
    scored: List[Dict[str, Any]] = []
    best_iv = cands[0]
    best_score = float("-inf")
    for iv in cands:
        avg = 0.0
        for _ in range(num_samples):
            sim = clone_state(state)
            apply_intervention(world, sim, iv)
            end = rollout(world, engine, sim, depth=depth)
            avg += score_state(world, end)
        avg /= num_samples
        scored.append({"move": iv.short(), "score": avg, "iv": iv})
        if avg > best_score:
            best_score = avg
            best_iv = iv

    if return_trace:
        trace = DMTrace(
            turn=state.turn,
            fired_nodes=[],   # filled in by the loop after engine.step
            classification=state.last_classification,
            candidates=[{"move": s["move"], "score": s["score"]} for s in scored],
            chosen=best_iv.short(),
            chosen_score=best_score,
            reason=explain_choice(world, state, best_iv),
            state_snapshot={
                "clues": list(state.discovered_clues),
                "executed": list(state.executed_nodes),
                "interventions": state.intervention_count,
                "denied": list(state.perm_denied_nodes),
                "temp_denied": dict(state.temp_denied_nodes),
                "location": state.player_location,
            },
        )
        return best_iv, trace
    return best_iv



## Step 7 — Intervention rendering

This step converts actions into prose. A small LLM call wraps the chosen intervention with language.

In [38]:
def item_for_clue(world: StoryWorld, clue_id: int) -> Optional[Item]:
    return next((it for it in world.game_world.items if it.related_clue_id == clue_id), None)


def room_rundown(world: StoryWorld, state: WorldState) -> str:
    here = world.game_world.get_location(state.player_location)
    parts: List[str] = []
    if here:
        parts.append(f"{state.player_location}: {here.description}")
    items_here = visible_items(world, state)
    npcs_here = visible_npcs(world, state)
    if items_here:
        parts.append(f"Items to examine: {', '.join(items_here)}.")
    if npcs_here:
        parts.append(f"People here: {', '.join(npcs_here)}.")
    if here and here.neighbors:
        parts.append(f"Exits: {', '.join(here.neighbors)}.")
    return "\n".join(parts)


def render_intervention(world: StoryWorld, state: WorldState, iv: Intervention) -> str:
    """Render DM interventions without leaking undiscovered clue contents.

    Hints should point the player toward a room or visible object, not reveal the
    clue text. The clue itself is only shown after an `examine` action succeeds.
    """
    if iv.kind == "noop":
        return ""

    if iv.kind == "glow":
        item = next((it for it in world.game_world.items if it.name == iv.target), None)
        if item and item.location == state.player_location:
            return f"Your attention catches on {item.name}. It may be worth examining."
        return ""

    if iv.kind == "hint":
        item = item_for_clue(world, int(iv.target))
        if not item:
            return ""
        if item.related_clue_id in state.discovered_clues:
            return ""
        if item.location == state.player_location:
            return f"Something about {item.name} still deserves a closer look."
        return ""

    if iv.kind == "cause":
        return "A new development opens another lead, but you still need evidence to make sense of it."

    if iv.kind in ("temp_deny", "perm_deny"):
        return "That lead will have to wait until you have firmer evidence."

    return ""


def discovered_clue_text(world: StoryWorld, clue_id: int) -> str:
    clue = world.get_clue(clue_id)
    if not clue:
        return ""
    return clue.description.strip()


def interview_text(world: StoryWorld, state: WorldState, name: str) -> str:
    char = world.get_character(name)
    if not char:
        return f"You interview {name}, but they add nothing useful."

    lines = [f"You interview {name}."]
    if char.description:
        lines.append(f"You note: {char.description}")
    if char.alibi:
        lines.append(f"Their account: {char.alibi}")
    else:
        lines.append("They give a cautious account of where they were during the murder.")
    lines.append("No clue is confirmed until you examine evidence that supports or contradicts the story.")
    return "\n".join(lines)


def ask_about_text(world: StoryWorld, state: WorldState, name: str, topic: str) -> str:
    char = world.get_character(name)
    if not char:
        return f"You ask {name} about {topic}, but get no useful answer."
    topic_clean = topic.strip() or "the case"
    lines = [f"You ask {name} about {topic_clean}."]
    if char.alibi:
        lines.append(f"They return to their main claim: {char.alibi}")
    else:
        lines.append("They answer carefully, but you need evidence before trusting it.")
    return "\n".join(lines)


def render_turn(
    world: StoryWorld,
    state: WorldState,
    action: Action,
    step_result: Dict[str, Any],
    iv_text: str,
) -> str:
    action = canonicalize_action(world, state, action)
    parts: List[str] = []
    if not step_result["ok"]:
        parts.append(step_result["message"])
    else:
        gw = world.game_world
        cls = step_result.get("classification", "")
        if action.verb == "move":
            parts.append(f"You move to {state.player_location}.")
            rundown = room_rundown(world, state)
            if rundown:
                parts.append(rundown)
        elif action.verb == "examine":
            it = next((i for i in gw.items if i.name == action.target), None)
            if it:
                parts.append(f"You examine {it.name}. {it.description}")
                if it.related_clue_id in state.discovered_clues:
                    clue_text = discovered_clue_text(world, it.related_clue_id)
                    if clue_text:
                        parts.append(f"Clue found: {clue_text}")
        elif action.verb == "interview":
            parts.append(interview_text(world, state, action.target))
        elif action.verb == "ask_about":
            parts.append(ask_about_text(world, state, action.target, action.indirect))
        elif action.verb == "use_item":
            if action.indirect:
                parts.append(f"You try using {action.target} on {action.indirect}, but nothing clear happens.")
            else:
                parts.append(f"You try using {action.target}, but nothing clear happens.")
        elif action.verb == "look":
            rundown = room_rundown(world, state)
            if rundown:
                parts.append(rundown)
        elif action.verb == "accuse":
            if cls == "case_solved":
                parts.append(f"You accuse {action.target}. The evidence holds. The case is closed.")
            else:
                parts.append(f"You accuse {action.target}, but you do not have enough proof. The case falls apart.")

        for nid in step_result["fired_nodes"]:
            node = next((n for n in world.plot_graph if n.id == nid), None)
            if node:
                parts.append(f"Story advanced: {node.title}.")

    if iv_text:
        parts.append(iv_text)
    return "\n".join(p for p in parts if p)






## Step 8 — Game Loop

Each turn:

1. Read free-text input.
2. Parse → structured action (LLM).
3. Engine validates and steps the world.
4. DM decides on an intervention (sampled rollouts).
5. Render the turn's narration.

Type `quit` to exit early; the game also ends on a correct accusation or after `max_turns`.

In [44]:
def format_state_panel(world: StoryWorld, state: WorldState) -> str:
    here = world.game_world.get_location(state.player_location)
    items_here = visible_items(world, state)
    npcs_here = visible_npcs(world, state)
    progress = f"{len(state.executed_nodes)}/{len(world.plot_graph)}"
    return (
        f"[STATE] turn={state.turn} loc={state.player_location} "
        f"clues={state.discovered_clues} nodes={progress} "
        f"interventions={state.intervention_count}\n"
        f"        items_here={items_here} npcs_here={npcs_here} "
        f"exits={(here.neighbors if here else [])}"
    )


def clean_sentence(text: str) -> str:
    return text.strip().rstrip(".")


def opening_briefing(world: StoryWorld, state: WorldState) -> str:
    cd = world.crime_details
    detective = next((c.name for c in world.characters if c.role == "detective"), "Detective")
    here = world.game_world.get_location(state.player_location)
    suspects = active_suspect_names(world, state)
    lines = [
        f"=== CASE: {cd.victim} ===",
        f"You are {detective}, investigating the {cd.crime_type} of {cd.victim}.",
        f"Scene: {cd.location}.",
        f"Goal: explore rooms, examine evidence, interview people, then accuse the killer when you have enough clues.",
    ]
    if world.countdown_deadline:
        lines.append(f"Deadline: {clean_sentence(world.countdown_deadline)}.")
    if world.countdown_consequence:
        lines.append(f"Stakes: {clean_sentence(world.countdown_consequence)}.")
    lines.append("")
    if here:
        lines.append(f"You start in {state.player_location}: {here.description}")
    items = visible_items(world, state)
    npcs = visible_npcs(world, state)
    if items:
        lines.append(f"Items here: {', '.join(items)}.")
    if npcs:
        lines.append(f"People here: {', '.join(npcs)}.")
    if here and here.neighbors:
        lines.append(f"Exits: {', '.join(here.neighbors)}.")
    if suspects:
        lines.append(f"Known suspects: {', '.join(suspects)}.")
    lines.extend([
        "",
        "Useful commands: look | examine <item> | move to <room> | interview <name> | ask <name> about <topic> | accuse <name> | quit",
        "Start with 'look' if you are unsure what to do.",
    ])
    return "\n".join(lines)


def play(world: StoryWorld, max_turns: int = 30, verbose: bool = False) -> WorldState:
    """Interactive REPL.

    When verbose=True, each turn prints a behind-the-scenes panel:
      - parsed action
      - engine classification + nodes that fired
      - DM candidates considered, top scores, chosen move + reason
      - narrated turn output
    The full per-turn DM trace is also saved on `state.dm_traces`.
    """
    engine = GameEngine(world)
    state = initialize_state(world)
    state.dm_traces = []  # type: ignore[attr-defined]
    print(opening_briefing(world, state))
    if verbose:
        print("\n(verbose mode: behind-the-scenes panels enabled)\n")

    while state.turn < max_turns and not state.case_solved:
        try:
            nl = input(">> ").strip()
        except EOFError:
            break
        if not nl:
            continue
        if nl.lower() in ("quit", "exit"):
            print("You walk away from the case.")
            break

        action = parse_action(world, state, nl)
        if verbose:
            print(f"\n[PARSE] verb={action.verb!r} target={action.target!r} indirect={action.indirect!r}")

        result = engine.step(state, action)
        action = canonicalize_action(world, state, action)
        if verbose:
            print(f"[ENGINE] ok={result['ok']} class={result['classification']} "
                  f"fired={result['fired_nodes']}")

        iv, trace = drama_manager_decide(world, engine, state, return_trace=True)
        trace.fired_nodes = result["fired_nodes"]
        trace.classification = result["classification"]
        state.dm_traces.append(trace)  # type: ignore[attr-defined]
        apply_intervention(world, state, iv)
        iv_text = render_intervention(world, state, iv) if iv.kind != "noop" else ""

        if verbose:
            print(trace.render())
            print(format_state_panel(world, state))
            print("\n[NARRATION]")

        print(render_turn(world, state, action, result, iv_text))
        print()

        if state.case_solved:
            print("*** Case closed. ***")
            break
        if result.get("classification") == "false_accusation":
            print("*** Wrong call. The case slips away. ***")
            break

    return state


def dump_dm_log(state: WorldState, path: str = "dm_log.json") -> None:
    """Persist the DM trace from a play()/run_eval() session for the demo video."""
    traces = getattr(state, "dm_traces", [])
    payload = []
    for t in traces:
        payload.append({
            "turn": t.turn,
            "classification": t.classification,
            "fired_nodes": t.fired_nodes,
            "chosen": t.chosen,
            "chosen_score": t.chosen_score,
            "reason": t.reason,
            "candidates": t.candidates,
            "state_snapshot": t.state_snapshot,
        })
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"Wrote {len(payload)} DM traces to {path}")


# Uncomment to play interactively (requires a working LLM API key):
state = play(world, verbose=False)
dump_dm_log(state, "dm_log.json")



=== CASE: Lord Alistair Finch, a notoriously ruthless real estate magnate and philanthropist ===
You are Detective Inspector Aris Thorne, investigating the murder of Lord Alistair Finch, a notoriously ruthless real estate magnate and philanthropist.
Scene: The Grand Ballroom of the Étoile Hotel, during the annual 'Starlight Charity Gala'.
Goal: explore rooms, examine evidence, interview people, then accuse the killer when you have enough clues.
Deadline: Lord Finch's grieving family, adhering to his known wishes for a swift and dignified process, has arranged for his body to be released from the morgue and embalmed by 12:00 PM tomorrow, in preparation for a private viewing and funeral service.
Stakes: If the detective fails to secure a court order for a full forensic toxicology examination before the embalming proceeds, the fast-acting Xylazine sedative, meticulously designed to mimic a natural heart attack, will be irrevocably flushed from Lord Finch's system by the embalming chemical

>>  look


Grand Ballroom: The opulent scene of Lord Finch's murder, now cordoned off by police.
Items to examine: Staff Interview Transcripts, Wait Staff Statement, Waiter's Statement.
People here: Victoria Sterling, Samuel Hayes, Dr. Eleanor Vance.
Exits: Hotel Lobby, Service Hallway.



>>  examine interview transcripts


You examine Staff Interview Transcripts. Transcripts detail Evelyn Reed's phone call in the secluded pantry.
Clue found: A temporary kitchen assistant reports seeing the lead caterer, Evelyn Reed, step away from the chaotic dessert station and make a quick, intense phone call in a secluded pantry for several minutes, precisely when Lord Finch was later found to have collapsed. This contradicts her claim of continuous oversight.



...

You move to Catering Kitchen.
Catering Kitchen: A large, chaotic kitchen still filled with equipment from the gala.
Items to examine: Silicone Ice Mold.
Exits: Service Hallway, Secluded Pantry, Head Caterer's Office.



>>  examine silicone ice mold


You examine Silicone Ice Mold. A peculiar silicone ice cube mold with a faint, unusual chemical odor.
Clue found: A slightly oversized, specialized silicone ice cube mold is discovered in a rarely used compartment within the catering kitchen's cold storage unit. A faint, unusual chemical odor lingers inside it.



>>  quit


You walk away from the case.
Wrote 5 DM traces to dm_log.json


## Step 9 — Evaluation Harness

Non-interactive auto-play with the metrics from the project plan:

- **turns_used** — proxy for story length
- **clues_discovered** — does the player actually uncover the mystery?
- **wasted_actions** — actions that neither advance a node nor reveal a clue
- **interventions** — manipulation budget consumed by the DM
- **executed_nodes** — fraction of the plot graph the player walked through

The auto-players (`scripted_player` and `random_player`) bypass the LLM action parser so as to avoid excessive API calls.

In [40]:
def shortest_path(gw: GameWorld, start: str, goal: str) -> List[str]:
    if start == goal:
        return [start]
    frontier: List[List[str]] = [[start]]
    seen = {start}
    while frontier:
        path = frontier.pop(0)
        here = gw.get_location(path[-1])
        if not here:
            continue
        for nxt in here.neighbors:
            if nxt in seen:
                continue
            new_path = path + [nxt]
            if nxt == goal:
                return new_path
            seen.add(nxt)
            frontier.append(new_path)
    return []


def move_toward(gw: GameWorld, start: str, goal: str) -> Action:
    path = shortest_path(gw, start, goal)
    if len(path) >= 2:
        return Action(verb="move", target=path[1])
    here = gw.get_location(start)
    if here and here.neighbors:
        return Action(verb="move", target=here.neighbors[0])
    return Action(verb="look")


def nearest_undiscovered_item(world: StoryWorld, state: WorldState) -> Optional[Item]:
    candidates = [it for it in world.game_world.items
                  if it.related_clue_id > 0 and it.related_clue_id not in state.discovered_clues]
    if not candidates:
        return None
    def distance(it: Item) -> int:
        path = shortest_path(world.game_world, state.player_location, it.location)
        return len(path) if path else 999
    return min(candidates, key=distance)


def nearest_uninterviewed_npc(world: StoryWorld, state: WorldState) -> Tuple[str, str]:
    candidates = [(name, loc) for name, loc in world.game_world.npc_locations.items()
                  if name not in state.interviewed]
    if not candidates:
        return "", ""
    def distance(pair: Tuple[str, str]) -> int:
        path = shortest_path(world.game_world, state.player_location, pair[1])
        return len(path) if path else 999
    return min(candidates, key=distance)


def clues_needed_to_accuse(world: StoryWorld) -> int:
    return max(3, len(world.clues) // 2)


def scripted_player(world: StoryWorld, state: WorldState) -> Action:
    """Deterministic successful player for the demo walkthrough.

    It gathers the minimum required evidence, then makes the correct accusation.
    This gives the final video one reliable solved run instead of depending on
    random walking through the map.
    """
    criminal = next((c.name for c in world.characters if c.is_criminal), "")
    if criminal and len(state.discovered_clues) >= clues_needed_to_accuse(world):
        return Action(verb="accuse", target=criminal)

    # First priority: collect clue items. Clues are what unlock a valid accusation.
    for it in world.game_world.items:
        if (it.location == state.player_location
                and it.related_clue_id > 0
                and it.related_clue_id not in state.discovered_clues):
            return Action(verb="examine", target=it.name)

    target_item = nearest_undiscovered_item(world, state)
    if target_item:
        return move_toward(world.game_world, state.player_location, target_item.location)

    # If all clue items are exhausted but the case still needs momentum, interview people.
    for name, loc in world.game_world.npc_locations.items():
        if loc == state.player_location and name not in state.interviewed:
            return Action(verb="interview", target=name)

    name, loc = nearest_uninterviewed_npc(world, state)
    if name:
        return move_toward(world.game_world, state.player_location, loc)

    if criminal:
        return Action(verb="accuse", target=criminal)
    return Action(verb="look")


def random_player(world: StoryWorld, state: WorldState) -> Action:
    gw = world.game_world
    options: List[Action] = []
    for it in gw.items:
        if it.location == state.player_location:
            options.append(Action(verb="examine", target=it.name))
    for name, loc in gw.npc_locations.items():
        if loc == state.player_location:
            options.append(Action(verb="interview", target=name))
    here = gw.get_location(state.player_location)
    if here:
        for n in here.neighbors:
            options.append(Action(verb="move", target=n))
    if not options:
        return Action(verb="look")
    return random.choice(options)


def run_eval(
    world: StoryWorld,
    player_fn: Callable[[StoryWorld, WorldState], Action],
    with_dm: bool = True,
    max_turns: int = 30,
    verbose: bool = False,
) -> Dict[str, Any]:
    """Auto-play with metrics. Set verbose=True to print the per-turn DM trace
    (useful for the final-video walkthrough); the trace is also stored on the
    returned state so you can dump_dm_log() afterward."""
    engine = GameEngine(world)
    state = initialize_state(world)
    state.dm_traces = []  # type: ignore[attr-defined]
    waste = 0
    for _ in range(max_turns):
        if state.case_solved:
            break
        action = player_fn(world, state)
        prev_clues = len(state.discovered_clues)
        result = engine.step(state, action)
        if result["ok"] and not result["fired_nodes"] and len(state.discovered_clues) == prev_clues:
            if action.verb not in ("accuse",):
                waste += 1
        if with_dm:
            iv, trace = drama_manager_decide(world, engine, state, num_samples=2, depth=3, return_trace=True)
            trace.fired_nodes = result["fired_nodes"]
            trace.classification = result["classification"]
            state.dm_traces.append(trace)  # type: ignore[attr-defined]
            apply_intervention(world, state, iv)
            if verbose:
                print(f"\nturn {state.turn}: action={action.verb}({action.target}) "
                      f"-> {result['classification']} fired={result['fired_nodes']}")
                print(trace.render())
        else:
            if verbose:
                print(f"turn {state.turn}: action={action.verb}({action.target}) "
                      f"-> {result['classification']} fired={result['fired_nodes']}  [DM disabled]")
    metrics = {
        "turns_used": state.turn,
        "case_solved": state.case_solved,
        "clues_discovered": len(state.discovered_clues),
        "executed_nodes": len(state.executed_nodes),
        "total_nodes": len(world.plot_graph),
        "interventions": state.intervention_count,
        "wasted_actions": waste,
    }
    return {"metrics": metrics, "state": state}

random.seed(0)
print("Scripted player + DM   :", run_eval(world, scripted_player, with_dm=True,  max_turns=35)["metrics"])
random.seed(0)
print("Scripted player no DM  :", run_eval(world, scripted_player, with_dm=False, max_turns=35)["metrics"])
random.seed(0)
print("Random player + DM     :", run_eval(world, random_player,   with_dm=True,  max_turns=25)["metrics"])
random.seed(0)
print("Random player no DM    :", run_eval(world, random_player,   with_dm=False, max_turns=25)["metrics"])

random.seed(0)
demo = run_eval(world, scripted_player, with_dm=True, max_turns=35, verbose=True)
dump_dm_log(demo["state"], "dm_log.json")


Scripted player + DM   : {'turns_used': 7, 'case_solved': True, 'clues_discovered': 4, 'executed_nodes': 4, 'total_nodes': 15, 'interventions': 3, 'wasted_actions': 2}
Scripted player no DM  : {'turns_used': 7, 'case_solved': True, 'clues_discovered': 4, 'executed_nodes': 2, 'total_nodes': 15, 'interventions': 0, 'wasted_actions': 2}
Random player + DM     : {'turns_used': 25, 'case_solved': False, 'clues_discovered': 4, 'executed_nodes': 2, 'total_nodes': 15, 'interventions': 8, 'wasted_actions': 21}
Random player no DM    : {'turns_used': 25, 'case_solved': False, 'clues_discovered': 4, 'executed_nodes': 1, 'total_nodes': 15, 'interventions': 0, 'wasted_actions': 21}

turn 1: action=examine(Staff Interview Transcripts) -> neutral fired=[]
[DM turn 1] classification=neutral fired=[]
  state: clues=[4] executed=[1] interventions=0
  considered 10 moves; top:
    +2.167  noop <-- chosen
    +2.167  hint(1)
    +2.167  hint(2)
    +2.167  hint(3)
    +2.167  hint(5)
  reason: player has 

## Phase II Walkthroughs

The cell below runs two deterministic auto-played walkthroughs of the game so we can see exactly what the system does at every
turn. Each turn is printed with two clearly labeled blocks:

- `[USER ACTION]` — the parsed structured action that was attempted, plus the engine's
  classification and any plot-graph nodes that fired as a result.
- `[DRAMA MANAGER ACTION]` — the intervention the DM chose for this turn, the top-3
  alternatives it considered with their scores, and a one-line reason explaining the
  choice.

We run two contrasting walkthroughs:

1. **Walkthrough A — Scripted player.** The heuristic player actively investigates
   (examines clues, interviews NPCs). The DM should mostly stay out of the way and
   intervene only when pacing or solvability would otherwise drift.
2. **Walkthrough B — Random player.** The player wanders randomly. The DM should be
   visibly more active — surfacing hints and causing near-ready nodes — to keep the
   case from stalling.

The full per-turn DM trace from each walkthrough is also written to disk
(`dm_log_walkthroughA.json`, `dm_log_walkthroughB.json`) for reference.


In [43]:
def walkthrough(world: StoryWorld, player_fn, label: str, max_turns: int = 12, log_path: str = "dm_log.json") -> WorldState:
    """Auto-played walkthrough that prints [USER ACTION] / [DRAMA MANAGER ACTION] blocks per turn.

    Deterministic when random.seed(...) is called first.
    """
    print("=" * 72)
    print(f"WALKTHROUGH — {label}")
    print("=" * 72)
    print(f"Crime: {world.crime_details.crime_type} of {world.crime_details.victim} at {world.crime_details.location}")
    print(f"Detective starts at: {world.game_world.starting_location}")
    print(f"Plot graph: {len(world.plot_graph)} nodes | clues: {len(world.clues)} | locations: {len(world.game_world.locations)}")
    print()

    engine = GameEngine(world)
    state = initialize_state(world)
    state.dm_traces = []  # type: ignore[attr-defined]

    for _ in range(max_turns):
        if state.case_solved:
            break

        action = player_fn(world, state)
        result = engine.step(state, action)

        # ----------- USER ACTION block -----------
        print(f"--- Turn {state.turn} ---")
        print(f"[USER ACTION] verb={action.verb!r} target={action.target!r} indirect={action.indirect!r}")
        print(f"              classification={result['classification']}  fired_nodes={result['fired_nodes']}  ok={result['ok']}")
        if not result["ok"] and result.get("message"):
            print(f"              engine_msg: {result['message']}")

        # ----------- DRAMA MANAGER ACTION block -----------
        iv, trace = drama_manager_decide(world, engine, state, num_samples=2, depth=3, return_trace=True)
        trace.fired_nodes = result["fired_nodes"]
        trace.classification = result["classification"]
        state.dm_traces.append(trace)  # type: ignore[attr-defined]

        top = sorted(trace.candidates, key=lambda x: -x["score"])[:3]
        top_str = " | ".join(f"{c['move']}={c['score']:+.2f}" for c in top)
        print(f"[DRAMA MANAGER ACTION] chose={trace.chosen} (score={trace.chosen_score:+.2f})")
        print(f"                       top3: {top_str}")
        print(f"                       reason: {trace.reason}")

        apply_intervention(world, state, iv)

        # State snapshot after the turn
        progress = f"{len(state.executed_nodes)}/{len(world.plot_graph)}"
        print(f"[STATE] location={state.player_location} clues={state.discovered_clues} "
              f"nodes={progress} interventions_used={state.intervention_count}")
        print()

        if state.case_solved:
            print("*** Case closed. ***")
            break
        if result.get("classification") == "false_accusation":
            print("*** Wrong call. The case slips away. ***")
            break

    # Persist the trace to disk
    dump_dm_log(state, log_path)
    print(f"[walkthrough done] turns={state.turn} solved={state.case_solved} "
          f"clues={len(state.discovered_clues)} interventions={state.intervention_count}")
    print()
    return state


# ---- Walkthrough A: deterministic successful player ----
random.seed(0)
state_A = walkthrough(world, scripted_player, "A — Scripted solver (successful)",
                      max_turns=35, log_path="dm_log_walkthroughA.json")

# ---- Walkthrough B: random wandering player, useful as an unsuccessful contrast ----
random.seed(7)
state_B = walkthrough(world, random_player, "B — Random wandering player",
                      max_turns=18, log_path="dm_log_walkthroughB.json")

# ---- Quick summary table of both walkthroughs ----
print("=" * 72)
print("WALKTHROUGH SUMMARY")
print("=" * 72)
def summarise(label, s):
    return (f"{label:30s}  turns={s.turn:2d}  solved={str(s.case_solved):5s}  "
            f"clues={len(s.discovered_clues):2d}/{len(world.clues):2d}  "
            f"nodes={len(s.executed_nodes):2d}/{len(world.plot_graph):2d}  "
            f"interventions={s.intervention_count:2d}")
print(summarise("Walkthrough A (scripted solver)", state_A))
print(summarise("Walkthrough B (random)",          state_B))



WALKTHROUGH — A — Scripted solver (successful)
Crime: murder of Lord Alistair Finch, a notoriously ruthless real estate magnate and philanthropist at The Grand Ballroom of the Étoile Hotel, during the annual 'Starlight Charity Gala'
Detective starts at: Grand Ballroom
Plot graph: 15 nodes | clues: 8 | locations: 8

--- Turn 1 ---
[USER ACTION] verb='examine' target='Staff Interview Transcripts' indirect=''
              classification=neutral  fired_nodes=[]  ok=True
[DRAMA MANAGER ACTION] chose=noop (score=+2.17)
                       top3: noop=+2.17 | hint(1)=+2.17 | hint(2)=+2.17
                       reason: player has agency; story is ahead of pace
[STATE] location=Grand Ballroom clues=[4] nodes=1/15 interventions_used=0

--- Turn 2 ---
[USER ACTION] verb='examine' target='Wait Staff Statement' indirect=''
              classification=neutral  fired_nodes=[]  ok=True
[DRAMA MANAGER ACTION] chose=cause(7) (score=+2.18)
                       top3: cause(7)=+2.18 | noop=+2.12 | h